In [ ]:
import os, sys, subprocess, re, urllib.request, gzip, shutil, warnings
import importlib.util

_PYTHON_DEPS = {
    "numpy": "numpy>=1.24", "pandas": "pandas>=2.0",
    "polars": "polars>=0.20", "pyarrow": "pyarrow>=14",
    "matplotlib": "matplotlib>=3.7",
    "scipy": "scipy>=1.10", "statsmodels": "statsmodels>=0.14",
}
_missing_deps = [package for module, package in _PYTHON_DEPS.items()
                 if importlib.util.find_spec(module) is None]
if _missing_deps:
    print("Installing missing Figure 4 dependencies:", " ".join(_missing_deps))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *_missing_deps], check=True)
    importlib.invalidate_caches()
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from pathlib import Path
_cwd = Path.cwd()
_repo_root = next((p for p in (_cwd, *_cwd.parents)
                   if (p / "scripts").is_dir()), _cwd)
if str(_repo_root) not in sys.path: sys.path.insert(0, str(_repo_root))
from scripts.common import config as repo_config, gcs

WORKDIR = repo_config.SUSIE_FINEMAP_DIR
os.makedirs(WORKDIR, exist_ok=True)
INPUT_CACHE = os.path.join(WORKDIR, "workspace_inputs")
os.makedirs(INPUT_CACHE, exist_ok=True)

def _workspace_input(relative, *, with_index=False):
    """Stage a bucket-prefixed input locally for pandas/cyvcf2."""
    relative = relative.lstrip("/")
    local = os.path.join(INPUT_CACHE, relative)
    remote = repo_config.workspace_bucket_path(relative)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        result = gcs.gcs_cp(remote, local)
        if result.returncode != 0:
            raise FileNotFoundError(f"Could not stage {remote}: {result.stderr}")
    if with_index and not os.path.exists(local + ".tbi"):
        result = gcs.gcs_cp(remote + ".tbi", local + ".tbi")
        if result.returncode != 0:
            raise FileNotFoundError(f"Could not stage {remote}.tbi: {result.stderr}")
    return local


META_TSV = (os.environ.get("AOU_FIGURE3_TASKS_TSV") or
            _workspace_input("phewas_meta/meta_analysis_results_genic_meff.tsv"))

SV_VCF_PATH = _workspace_input(
    "SV_10k/AoU12k.vs.AoU-HPRC-v1.1b.AF_gt_pt005."
    "25kb_windows.0d0e0eb4.kage.glimpse.collisionless.merged.vcf.gz",
    with_index=True)

DISC_WORKDIR     = "phewas_sv_v5"
DISC_COV         = _workspace_input(f"{DISC_WORKDIR}/covariates_base.tsv")
DISC_PCA         = _workspace_input(f"{DISC_WORKDIR}/pca_scores_afr.tsv.bgz")
DISC_PHE         = _workspace_input(f"{DISC_WORKDIR}/aou_phecodeX_counts.tsv")
FLAGGED_IDS_PATH = _workspace_input(f"{DISC_WORKDIR}/relatedness_flagged_in_cohort.tsv")

PHECODE_SEX_PATH = repo_config.ASSETS_DIR / "phecodeX_sex.csv"

HARDMASK_PATH = _workspace_input("hardmask.hg38.v4.over99.bed")
GTF_PATH      = _workspace_input("hg38.refGene.gtf")
GTF_CACHE     = "hg38.refGene.exons.parquet"
GTF_URL       = ("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/"
                 "bigZips/genes/hg38.refGene.gtf.gz")

WINDOW            = 1_000_000
L_SUSIE           = 10
MIN_MAC           = 5
MIN_CARRIERS      = 20
MAX_MISS          = 0.05
MIN_CASES         = 1
MIN_PHECODE_COUNT = 2
SV_LEN_DIFF       = 50
N_PCS_DISC        = 20
ANALYSIS_CONTRACT = "10k_disc_20pc_sexaware_1mb_carriers20_v4"
USE_COVARIATES    = True
MAX_TASKS         = None
FONT = 34
SUPPLEMENTARY_GENE_FONTSIZE = 26
COLORBAR_VERTICAL_HEADROOM = 0.04
_SMALL_RNA_RE = re.compile(
    r"^(MIR|MIRLET|SNOR|SNORD|SNORA|SCARNA|RNU|RN7|RNA5|VTRNA|RNY)",
    re.IGNORECASE,
)
plt.rcParams.update({
    "font.size": FONT, "axes.titlesize": FONT, "axes.labelsize": FONT,
    "xtick.labelsize": FONT, "ytick.labelsize": FONT, "legend.fontsize": FONT,
    "figure.titlesize": FONT,
})

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    from cyvcf2 import VCF as CyVCF
except ImportError:
    _pip("cyvcf2"); from cyvcf2 import VCF as CyVCF

from scripts.susie_rscript import ensure_susie_r, run_susie
ensure_susie_r()

def _short_variant_label(vid, is_sv, sv_type):
    """Return a compact label for a variant, especially SVs."""
    parts = vid.split(":")
    if len(parts) < 2:
        return vid
    pos_str = f"{parts[0]}:{parts[1]}"
    if is_sv and len(parts) >= 4:
        try:
            svlen = abs(int(parts[3]))
            return f"{pos_str} {svlen}bp {sv_type.lower()}"
        except ValueError:
            return f"{pos_str} {sv_type.lower()}"
    elif len(parts) >= 4:
        ref, alt = parts[2], parts[3]
        if len(ref) > 5 or len(alt) > 5:
            diff = len(alt) - len(ref)
            if diff > 0:
                return f"{pos_str} {diff}bp ins"
            elif diff < 0:
                return f"{pos_str} {-diff}bp del"
            else:
                return f"{pos_str} {len(ref)}bp sub"
        return vid
    return vid

print("=" * 72)
print("STAGE 0 — Load meta-significant SNV leads")
print("=" * 72)

with open(META_TSV) as fh:
    header_row = None
    for i, line in enumerate(fh):
        if "variant_tag" in line.split("\t")[0]:
            header_row = i
            break
if header_row is None:
    raise RuntimeError(f"No 'variant_tag' header row in {META_TSV}")

print(f"  Reading {META_TSV}  (header at line {header_row+1})")
raw = pd.read_csv(META_TSV, sep="\t", skiprows=header_row, dtype=str)
raw.columns = raw.columns.str.strip()

raw = raw.dropna(subset=["variant_tag", "phecode", "meta_significant"])
raw = raw[raw["meta_significant"].str.strip().str.upper() == "TRUE"].copy()
print(f"  meta_significant rows: {len(raw)}")

def _parse_snv_tag(tag):
    parts = str(tag).strip().split("_", 3)
    if len(parts) < 4:
        return None, None, None, None
    try:
        return int(parts[0]), int(parts[1]), parts[2], parts[3]
    except Exception:
        return None, None, None, None

raw[["CHR", "POS", "REF", "ALT"]] = raw["variant_tag"].apply(
    lambda t: pd.Series(_parse_snv_tag(t)))
raw = raw.dropna(subset=["CHR", "POS"]).copy()
raw["CHR"] = raw["CHR"].astype(int)
raw["POS"] = raw["POS"].astype(int)
raw["phecode"]      = raw["phecode"].astype(str).str.strip()
raw["phecode_name"] = (raw["phecode_string"].astype(str).str.strip()
                       if "phecode_string" in raw.columns else raw["phecode"])

for c in ("beta_d", "se_d", "p_d", "n_cases_d", "n_controls_d", "mac_disc",
          "beta_r", "se_r", "p_r", "n_cases_r", "n_controls_r", "mac_rep",
          "beta_meta", "se_meta", "p_meta"):
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

tasks = (raw.drop_duplicates(subset=["variant_tag", "phecode"], keep="first")
            .reset_index(drop=True))
if MAX_TASKS is not None:
    tasks = tasks.head(MAX_TASKS).reset_index(drop=True)

print(f"  Tasks to fine-map: {len(tasks)}")
for ti, t in tasks.iterrows():
    gene_str = str(t.get("gene_in_window", "")) if pd.notna(t.get("gene_in_window")) else ""
    print(f"    [{ti+1}] chr{t.CHR}:{t.POS} {t.REF}>{t.ALT}  "
          f"{t.phecode} {t.phecode_name}  ({gene_str})")

_bed = pl.scan_csv(
    HARDMASK_PATH, separator="\t", has_header=False,
    new_columns=["chrom", "start", "end"],
    schema_overrides={"chrom": pl.Utf8, "start": pl.Int64, "end": pl.Int64},
)

def make_hardmask_checker(chrom, start, end):
    intervals = _bed.filter(
        (pl.col("chrom") == f"chr{chrom}")
        & (pl.col("end")   > start - 1)
        & (pl.col("start") < end)
    ).collect()
    if intervals.height == 0:
        return lambda p: False
    starts = intervals["start"].to_numpy()
    ends   = intervals["end"].to_numpy()
    def check(pos):
        p0 = pos - 1
        return bool(np.any((starts <= p0) & (ends > p0)))
    return check

def load_gtf_exons():
    if os.path.exists(GTF_CACHE):
        return pl.read_parquet(GTF_CACHE)
    if not os.path.exists(GTF_PATH):
        print(f"  Downloading {GTF_URL} → {GTF_PATH} …")
        gz_path = GTF_PATH + ".gz"
        urllib.request.urlretrieve(GTF_URL, gz_path)
        with gzip.open(gz_path, "rb") as f_in, open(GTF_PATH, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)
    print(f"  Parsing {GTF_PATH} (one-time) …")
    df = pl.read_csv(
        GTF_PATH, separator="\t", has_header=False, comment_prefix="#",
        new_columns=["chrom", "source", "feature", "start", "end", "score",
                     "strand", "frame", "attribute"],
        schema_overrides={"chrom": pl.Utf8, "start": pl.Int64, "end": pl.Int64,
                          "strand": pl.Utf8, "attribute": pl.Utf8,
                          "score": pl.Utf8, "frame": pl.Utf8},
    ).filter(pl.col("feature") == "exon")
    df = df.with_columns([
        pl.col("attribute").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_name"),
        pl.col("attribute").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    ]).drop("attribute", "source", "frame", "score", "feature")
    df.write_parquet(GTF_CACHE)
    return df

GTF_EXONS = load_gtf_exons()
print(f"  GTF: {GTF_EXONS.height:,} exons across "
      f"{GTF_EXONS.select(pl.col('gene_name').n_unique()).item():,} genes")

def genes_in_region(chrom, start, end):
    sub = GTF_EXONS.filter(
        (pl.col("chrom") == f"chr{chrom}")
        & (pl.col("end") >= start) & (pl.col("start") <= end))
    if sub.height == 0:
        return pd.DataFrame(columns=["gene_name", "transcript_id", "strand",
                                      "tx_start", "tx_end", "span", "exons"])
    tx_span = sub.group_by(["gene_name", "transcript_id", "strand"]).agg([
        pl.col("start").min().alias("tx_start"),
        pl.col("end").max().alias("tx_end"),
    ]).with_columns((pl.col("tx_end") - pl.col("tx_start")).alias("span"))
    longest = (tx_span.sort("span", descending=True)
                       .group_by("gene_name").first()).to_pandas()
    exons_by_tx = {}
    for tid in longest["transcript_id"]:
        ex = sub.filter(pl.col("transcript_id") == tid)
        exons_by_tx[tid] = list(zip(ex["start"].to_list(), ex["end"].to_list()))
    longest["exons"] = longest["transcript_id"].map(exons_by_tx)
    return longest.sort_values("tx_start").reset_index(drop=True)

def draw_gene_track(ax, chrom, region_start, region_end, max_genes=None):
    genes = genes_in_region(chrom, region_start, region_end)
    genes = genes[~genes["gene_name"].str.match(_SMALL_RNA_RE)]
    genes = genes[genes["span"] >= 1_000]
    if len(genes) == 0:
        ax.text(0.5, 0.5, "(no annotated genes)", transform=ax.transAxes,
                ha="center", va="center", color="gray",
                fontsize=SUPPLEMENTARY_GENE_FONTSIZE)
        ax.set_yticks([]); ax.set_xlim(region_start / 1e6, region_end / 1e6)
        return
    if max_genes is not None and len(genes) > max_genes:
        genes = (genes.sort_values("span", ascending=False).head(max_genes)
                       .sort_values("tx_start").reset_index(drop=True))
    min_gap = (region_end - region_start) * 0.24
    rows, row_of = [], []
    for _, g in genes.iterrows():
        placed = False
        for ri, last in enumerate(rows):
            if g["tx_start"] > last + min_gap:
                rows[ri] = g["tx_end"]; row_of.append(ri); placed = True; break
        if not placed:
            rows.append(g["tx_end"]); row_of.append(len(rows) - 1)
    row_height = 0.6
    row_gap = 3.0
    ax.set_ylim(-0.7, (len(rows) - 1) * row_gap + 2.8)
    min_exon_w = (region_end - region_start) * 0.002
    for (_, g), row in zip(genes.iterrows(), row_of):
        y = row * row_gap + 0.5
        ax.hlines(y, g["tx_start"] / 1e6, g["tx_end"] / 1e6,
                  colors="black", linewidth=1.8, zorder=2)
        for ex_s, ex_e in g["exons"]:
            es = max(ex_s, region_start); ee = min(ex_e, region_end)
            if ee <= es: continue
            ax.add_patch(plt.Rectangle(
                (es / 1e6, y - row_height / 2),
                max((ee - es) / 1e6, min_exon_w / 1e6), row_height,
                facecolor="black", edgecolor="none", alpha=0.85, zorder=3))
        center_x = np.clip((g["tx_start"] + g["tx_end"]) / 2,
                           region_start + (region_end - region_start) * 0.05,
                           region_end - (region_end - region_start) * 0.05) / 1e6
        ax.text(center_x, y + row_height / 2 + 0.35, g['gene_name'],
                ha="center", va="bottom",
                fontsize=SUPPLEMENTARY_GENE_FONTSIZE,
                fontstyle="italic", color="black", zorder=4)
    ax.set_xlim(region_start / 1e6, region_end / 1e6)
    ax.set_ylabel("Genes")
    ax.set_yticks([])
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

def _id_col(columns):
    for c in ("sample_id", "person_id", "research_id", "s", "IID"):
        if c in columns: return c
    return columns[0]

def _harmonize_pca(df_raw, n_pcs):
    df = df_raw.copy()
    df.rename(columns={_id_col(list(df.columns)): "person_id"}, inplace=True)
    df["person_id"] = df["person_id"].astype(int)
    _arr_col = None
    for candidate in ("pca_features", "scores"):
        if candidate in df.columns and (
                pd.api.types.is_object_dtype(df[candidate])
                or pd.api.types.is_string_dtype(df[candidate])):
            _arr_col = candidate; break
    if _arr_col is not None:
        exp = (df[_arr_col].astype(str).str.strip()
               .str.strip("[]").str.split(",", expand=True))
        for i in range(n_pcs):
            df[f"pc{i+1}"] = pd.to_numeric(exp[i], errors="coerce")
        df.drop(columns=[_arr_col], inplace=True, errors="ignore")
        for _drop in ("ancestry_pred", "probabilities", "ancestry_pred_other"):
            df.drop(columns=[_drop], inplace=True, errors="ignore")
    else:
        up = {c.upper(): c for c in df.columns if c != "person_id"}
        remap = {}
        for i in range(1, n_pcs + 1):
            for pat in [f"PC{i}", f"SCORES_{i-1}", f"SCORES_{i}",
                        f"PCA_FEATURE_{i}", f"PCA{i}", f"COMPONENT_{i}"]:
                if pat in up: remap[up[pat]] = f"pc{i}"; break
        if len(remap) < n_pcs:
            num_cols = [c for c in df.columns if c != "person_id"
                        and pd.api.types.is_numeric_dtype(df[c])]
            if len(num_cols) >= n_pcs:
                for i, c in enumerate(num_cols[:n_pcs]):
                    remap[c] = f"pc{i+1}"
        df.rename(columns=remap, inplace=True)
    return df[["person_id"] + [f"pc{i}" for i in range(1, n_pcs + 1)]]

def fetch_region_cyvcf2(vcf_path, chrom, start, end, n_samples, sample_indices=None):
    vcf = CyVCF(vcf_path)
    hm  = make_hardmask_checker(chrom, start, end)
    ids, pos_list, dosages, carrier_masks = [], [], [], []
    is_sv_list, sv_type_list = [], []
    for rec in vcf(f"chr{chrom}:{max(1, start)}-{end}"):
        if rec.ALT is None or len(rec.ALT) != 1: continue
        gt  = rec.gt_types
        dos = np.where(gt == 2, np.nan, np.where(gt == 3, 2.0, gt.astype(float)))
        if sample_indices is not None:
            dos = dos[np.asarray(sample_indices, dtype=int)]
        else:
            dos = dos[:n_samples]
        if len(dos) != n_samples:
            raise ValueError(f"Expected {n_samples} genotype columns, found {len(dos)}")
        nz = ~np.isnan(dos)
        an = 2 * int(nz.sum())
        if an == 0 or (1.0 - nz.mean()) > MAX_MISS: continue
        ac = float(np.nansum(dos)); mac = min(ac, an - ac)
        if mac < MIN_MAC: continue
        observed_carriers = nz & (dos > 0)
        if np.isnan(dos).any(): dos = np.where(nz, dos, np.nanmean(dos))
        ref, alt = rec.REF, rec.ALT[0]
        diff = len(ref) - len(alt)
        if diff >= SV_LEN_DIFF:      is_sv, sv_type = True, "DEL"
        elif -diff >= SV_LEN_DIFF:   is_sv, sv_type = True, "INS"
        else:                         is_sv, sv_type = False, ""
        if (not is_sv) and len(ref) == 1 and len(alt) == 1 and hm(rec.POS): continue
        if is_sv:
            svlen = rec.INFO.get("SVLEN", "")
            vid   = f"chr{chrom}:{rec.POS}:{sv_type}:{svlen}"
        else:
            vid = f"chr{chrom}:{rec.POS}:{ref}:{alt}"
        ids.append(vid); pos_list.append(rec.POS)
        dosages.append(dos.astype(np.float32))
        carrier_masks.append(observed_carriers)
        is_sv_list.append(bool(is_sv)); sv_type_list.append(sv_type)
    vcf.close()
    if not dosages:
        empty = np.zeros((n_samples, 0), dtype=np.float32)
        return [], empty, empty.astype(bool), [], [], []
    return (ids, np.stack(dosages, axis=1), np.stack(carrier_masks, axis=1),
            pos_list, is_sv_list, sv_type_list)

pc_counts = pl.read_csv(DISC_PHE, separator="\t",
                        schema_overrides={"phecode": pl.Utf8})
phecode_sex_df = pd.read_csv(PHECODE_SEX_PATH, dtype=str)
phecode_sex_df = phecode_sex_df.drop_duplicates("phecode")
PHECODE_SEX = dict(zip(phecode_sex_df["phecode"].str.strip(),
                        phecode_sex_df["sex"].str.strip()))

def phecode_sex_scope(phecode):
    scope = PHECODE_SEX.get(str(phecode).strip())
    if scope not in {"Both", "Female", "Male"}:
        raise KeyError(f"No valid sex definition for phecode {phecode!r}")
    return scope

def build_phenotype(phecode, sample_ids_int, has_cov_mask, sex_at_birth):
    cases = set(pc_counts.filter(
        (pl.col("phecode") == phecode)
        & (pl.col("count")  >= MIN_PHECODE_COUNT)
    )["person_id"].to_list())
    any_hit = set(pc_counts.filter(
        pl.col("phecode") == phecode
    )["person_id"].to_list())
    n = len(sample_ids_int)
    y = np.full(n, np.nan, dtype=float)
    in_case = np.fromiter((pid in cases   for pid in sample_ids_int),
                          dtype=bool, count=n)
    in_any  = np.fromiter((pid in any_hit for pid in sample_ids_int),
                          dtype=bool, count=n)
    y[in_case] = 1.0
    y[(~in_case) & (~in_any)] = 0.0
    scope = phecode_sex_scope(phecode)
    if scope == "Female":
        y[np.asarray(sex_at_birth) != 0] = np.nan
    elif scope == "Male":
        y[np.asarray(sex_at_birth) != 1] = np.nan
    if USE_COVARIATES: y[~has_cov_mask] = np.nan
    return y

def residualize(y, Z):
    Z1 = np.column_stack([np.ones(len(y)), Z])
    coef, *_ = np.linalg.lstsq(Z1, y, rcond=None)
    return y - Z1 @ coef

def score_test_region(Xm, ym, Zm):
    Z1 = np.column_stack([np.ones(len(ym)), Zm])
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", message="Inverting hessian failed")
        try:
            null_fit = sm.Logit(ym, Z1).fit(disp=False, maxiter=500, method="newton")
        except np.linalg.LinAlgError:
            null_fit = sm.Logit(ym, Z1).fit(disp=False, maxiter=1000, method="bfgs")
    p_null   = null_fit.predict(Z1)
    W        = p_null * (1 - p_null)
    r_null   = ym - p_null
    WZ       = Z1 * W[:, None]
    ZtWZ_inv = np.linalg.pinv(Z1.T @ WZ)
    xtwx     = (Xm * (Xm * W[:, None])).sum(axis=0)
    xtwz     = Xm.T @ WZ
    adj      = np.einsum("pk,kl,pl->p", xtwz, ZtWZ_inv, xtwz)
    info     = xtwx - adj
    U        = Xm.T @ r_null
    valid    = info > 0
    chi2     = np.where(valid, (U ** 2) / np.where(valid, info, np.nan), np.nan)
    return (np.where(valid, U / np.where(valid, info, np.nan), np.nan),
            np.where(valid, 1.0 / np.sqrt(np.where(valid, info, np.nan)), np.nan),
            stats.chi2.sf(chi2, df=1))

def wald_logit_single(x, y, Z):
    design = sm.add_constant(np.column_stack([x, Z]))
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", message="Inverting hessian failed")
        try:
            fit = sm.Logit(y, design).fit(disp=False, maxiter=300, method="newton")
        except Exception:
            try:
                fit = sm.Logit(y, design).fit(disp=False, maxiter=500, method="bfgs")
            except Exception:
                return _fisher_exact(x, y)
    b = float(fit.params[1]); se = float(fit.bse[1]); p = float(fit.pvalues[1])
    if not np.isfinite(se) or not np.isfinite(p): return _fisher_exact(x, y)
    return dict(beta=b, se=se, p=p, or_=float(np.exp(b)),
                n=int(fit.nobs), method="wald")

def _fisher_exact(x, y):
    carrier, case = x > 0, y > 0
    a = int((carrier & case).sum());  b = int((carrier & ~case).sum())
    c = int((~carrier & case).sum()); d = int((~carrier & ~case).sum())
    or_, p = stats.fisher_exact([[a, b], [c, d]])
    return dict(or_=float(or_), p=float(p), n=int(len(y)),
                a=a, b=b, c=c, d=d, method="fisher")

def variant_allele_stats(dos):
    n = len(dos); n_carriers = int((dos > 0).sum()); n_hom = int((dos == 2).sum())
    mac = 2 * n_hom + (n_carriers - n_hom)
    af  = mac / (2 * n) if n else float("nan")
    return n_carriers, mac, af

def compute_r2_to_lead_dosage(Xm, lead_idx):
    if lead_idx is None:
        return np.full(Xm.shape[1], np.nan)
    Xc = Xm - Xm.mean(axis=0)
    lc = Xc[:, lead_idx]
    num   = Xc.T @ lc
    denom = np.sqrt((Xc ** 2).sum(axis=0) * (lc ** 2).sum())
    r2    = (num / np.where(denom > 0, denom, np.nan)) ** 2
    return r2

def _build_ref_info(var_ids, positions, is_sv_list, sv_type_list, idx):
    """Build a ref_info dict for haplotype r² computation from a given variant index."""
    vid_parts = var_ids[idx].split(":")
    pos_val = positions[idx]
    if is_sv_list[idx]:
        svlen = 0
        if len(vid_parts) >= 4:
            try:
                svlen = abs(int(vid_parts[3]))
            except ValueError:
                pass
        return {"pos": pos_val, "is_sv": True,
                "sv_type": sv_type_list[idx], "sv_len": svlen}
    else:
        ref = vid_parts[2] if len(vid_parts) >= 3 else ""
        alt = vid_parts[3] if len(vid_parts) >= 4 else ""
        return {"pos": pos_val, "is_sv": False, "ref": ref, "alt": alt}

def _fetch_haps_bcftools(panel_vcf, chrom, start, end):
    cmd = ["bcftools", "query",
           "-r", f"chr{chrom}:{max(0, start)}-{end}",
           "-f", "%POS\t%REF\t%ALT[\t%GT]\n",
           str(panel_vcf)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True,
                           check=True, timeout=300)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError):
        return None
    if not r.stdout.strip():
        return None
    pos_l, ref_l, alt_l, hap_l = [], [], [], []
    for line in r.stdout.split("\n"):
        if not line: continue
        parts = line.split("\t")
        if len(parts) < 4: continue
        pos = int(parts[0]); ref = parts[1]; alt = parts[2]
        if "," in alt: continue
        gts = parts[3:]
        haps = np.empty(len(gts) * 2, dtype=np.int8)
        ok = True
        for i, g in enumerate(gts):
            if len(g) < 3 or g[0] == "." or g[2] == ".":
                ok = False; break
            haps[2*i]     = 1 if g[0] == "1" else 0
            haps[2*i + 1] = 1 if g[2] == "1" else 0
        if not ok: continue
        pos_l.append(pos); ref_l.append(ref); alt_l.append(alt)
        hap_l.append(haps)
    if not hap_l: return None
    return (np.asarray(pos_l, dtype=np.int64),
            [str(r) for r in ref_l],
            [str(a) for a in alt_l],
            np.asarray(hap_l, dtype=np.int8))

def compute_r2_haplotype(panel_vcf, chrom, win_start, win_end,
                         ref_info, var_ids, positions,
                         is_sv_list, sv_type_list):
    n_var = len(var_ids)
    r2_out = np.full(n_var, np.nan)
    if ref_info is None: return r2_out
    fetched = _fetch_haps_bcftools(panel_vcf, chrom, win_start, win_end)
    if fetched is None:
        print("      ⚠ bcftools haplotype extraction failed; r² will be NaN")
        return r2_out
    h_pos, h_refs, h_alts, H = fetched
    n_haps = H.shape[1]
    ref_pos   = ref_info["pos"]
    ref_is_sv = ref_info.get("is_sv", False)
    ref_h_idx = None
    if ref_is_sv:
        sv_len = ref_info.get("sv_len", 0)
        for i in range(len(h_pos)):
            if abs(int(h_pos[i]) - ref_pos) > 100: continue
            rl, al = len(h_refs[i]), len(h_alts[i])
            if abs(rl - al) < SV_LEN_DIFF: continue
            vid_svlen = abs(rl - al)
            if sv_len > 0 and vid_svlen > 0:
                if abs(vid_svlen - sv_len) / max(sv_len, 1) > 0.10: continue
            ref_h_idx = i; break
    else:
        ref_ref = ref_info.get("ref", "")
        ref_alt = ref_info.get("alt", "")
        for i in range(len(h_pos)):
            if int(h_pos[i]) != ref_pos: continue
            if h_refs[i] == ref_ref and h_alts[i] == ref_alt:
                ref_h_idx = i; break
    if ref_h_idx is None:
        kind = "SV" if ref_is_sv else "SNV"
        print(f"      ⚠ ref {kind} pos={ref_pos} not found in panel haplotypes")
        return r2_out
    H_f = H.astype(np.float64)
    AF  = H_f.sum(axis=1) / n_haps
    ref_hap = H_f[ref_h_idx]
    ref_af  = AF[ref_h_idx]
    pAB   = (H_f @ ref_hap) / n_haps
    D     = pAB - AF * ref_af
    denom = np.sqrt(AF * (1.0 - AF) * ref_af * (1.0 - ref_af))
    with np.errstate(divide="ignore", invalid="ignore"):
        r_vec = np.where(denom > 0, D / denom, 0.0)
    r2_hap = r_vec ** 2
    hap_lookup_snv = {}
    hap_lookup_sv  = {}
    for i in range(len(h_pos)):
        rl, al = len(h_refs[i]), len(h_alts[i])
        if rl == 1 and al == 1:
            hap_lookup_snv[(int(h_pos[i]), h_refs[i], h_alts[i])] = i
        elif abs(rl - al) >= SV_LEN_DIFF:
            hap_lookup_sv.setdefault(int(h_pos[i]), []).append(
                (i, abs(rl - al)))
    for j in range(n_var):
        p = positions[j]
        vid_parts = var_ids[j].split(":")
        if not is_sv_list[j]:
            if len(vid_parts) >= 4:
                key = (p, vid_parts[2], vid_parts[3])
                hi = hap_lookup_snv.get(key)
                if hi is not None:
                    r2_out[j] = float(r2_hap[hi])
        else:
            cands = hap_lookup_sv.get(p, [])
            if not cands:
                for dp in (-1, 1):
                    cands = hap_lookup_sv.get(p + dp, [])
                    if cands: break
            vid_svlen = 0
            if len(vid_parts) >= 4:
                try: vid_svlen = abs(int(vid_parts[3]))
                except ValueError: pass
            for hi, h_svlen in cands:
                if vid_svlen > 0 and h_svlen > 0:
                    if abs(vid_svlen - h_svlen) / max(h_svlen, 1) > 0.10: continue
                r2_out[j] = float(r2_hap[hi]); break
    n_mapped = int(np.isfinite(r2_out).sum())
    kind = "SV" if ref_is_sv else "SNV"
    print(f"      haplotype r² to {kind}@{ref_pos}: "
          f"{n_mapped}/{n_var} variants mapped  (ref AF={ref_af:.3f})")
    return r2_out

print("\n" + "=" * 72)
print("STAGE 1 — 10k disc cohort setup")
print("=" * 72)

assert os.path.exists(SV_VCF_PATH), f"SV VCF not found: {SV_VCF_PATH}"
_v0 = CyVCF(SV_VCF_PATH)
DISC_SAMPLES = list(_v0.samples); _v0.close()
N_DISC       = len(DISC_SAMPLES)
DISC_INT     = np.array([int(s) for s in DISC_SAMPLES], dtype=np.int64)
print(f"  SV VCF samples: {N_DISC}")

DISC_COV_CACHE = os.path.join(WORKDIR, "disc_covariates.tsv")
if os.path.exists(DISC_COV_CACHE):
    print(f"  Loading cached disc covariates: {DISC_COV_CACHE}")
    disc_cov_pd = pd.read_csv(DISC_COV_CACHE, sep="\t")
    disc_cov_pd["person_id"] = disc_cov_pd["person_id"].astype(int)
else:
    print("  Building disc covariates …")
    cov_base = pd.read_csv(DISC_COV, sep="\t")
    cov_base.rename(columns={_id_col(list(cov_base.columns)): "person_id"}, inplace=True)
    cov_base["person_id"] = cov_base["person_id"].astype(int)
    pca_raw = pd.read_csv(DISC_PCA, sep="\t", compression="gzip")
    pca_df  = _harmonize_pca(pca_raw, N_PCS_DISC)
    disc_cov_pd = cov_base.merge(pca_df, on="person_id", how="inner")
    if "age_at_last_ehr_event_squared" not in disc_cov_pd.columns:
        disc_cov_pd["age_at_last_ehr_event_squared"] = disc_cov_pd["age_at_last_ehr_event"] ** 2

    if os.path.exists(FLAGGED_IDS_PATH):
        flagged_df  = pd.read_csv(FLAGGED_IDS_PATH, sep="\t")
        FLAGGED_IDS = set(pd.to_numeric(flagged_df.iloc[:, 0], errors="coerce")
                          .dropna().astype(int).tolist())
        disc_cov_pd = disc_cov_pd[~disc_cov_pd["person_id"].isin(FLAGGED_IDS)]
        print(f"    excluded {len(FLAGGED_IDS)} flagged related samples")
    DISC_PC_COLS  = [f"pc{i}" for i in range(1, N_PCS_DISC + 1)]
    DISC_COV_COLS = ["age_at_last_ehr_event", "age_at_last_ehr_event_squared",
                     "sex_at_birth"] + DISC_PC_COLS
    disc_cov_pd = disc_cov_pd.dropna(subset=DISC_COV_COLS)

DISC_PC_COLS  = [f"pc{i}" for i in range(1, N_PCS_DISC + 1)]
DISC_COV_COLS = ["age_at_last_ehr_event", "age_at_last_ehr_event_squared",
                 "sex_at_birth"] + DISC_PC_COLS
missing_disc_cov = [c for c in DISC_COV_COLS if c not in disc_cov_pd.columns]
if missing_disc_cov:
    raise ValueError(f"Discovery covariates missing required columns: {missing_disc_cov}")
sex_raw = disc_cov_pd["sex_at_birth"]
if sex_raw.dtype == object:
    normalized = sex_raw.astype(str).str.strip().str.lower()
    normalized = normalized.replace({"male": "1", "m": "1",
                                     "female": "0", "f": "0"})
    disc_cov_pd["sex_at_birth"] = pd.to_numeric(normalized, errors="coerce")
else:
    disc_cov_pd["sex_at_birth"] = pd.to_numeric(sex_raw, errors="coerce")
invalid_sex = ~disc_cov_pd["sex_at_birth"].isin([0, 1])
if invalid_sex.any():
    raise ValueError(f"sex_at_birth must be female=0/male=1; invalid rows={invalid_sex.sum()}")
disc_cov_pd = disc_cov_pd.dropna(subset=DISC_COV_COLS)
disc_cov_pd.to_csv(DISC_COV_CACHE, sep="\t", index=False)
print(f"  ✓ discovery covariates cached → {DISC_COV_CACHE}")
print(f"  discovery covariates: {len(disc_cov_pd)} samples, {N_PCS_DISC} PCs")
disc_cov_pd = disc_cov_pd.set_index("person_id")
disc_cov_pd.index = disc_cov_pd.index.astype(str)

disc_cov_index = np.full(N_DISC, -1, dtype=np.int64)
_pos_in_cov = {pid: i for i, pid in enumerate(disc_cov_pd.index.tolist())}
for i, s in enumerate(DISC_SAMPLES):
    j = _pos_in_cov.get(s)
    if j is not None: disc_cov_index[i] = j
DISC_HAS_COV = disc_cov_index >= 0
_disc_cov_rows = np.where(DISC_HAS_COV)[0]
DISC_COV_MAT = np.full((N_DISC, len(DISC_COV_COLS)), np.nan, dtype=float)
DISC_COV_MAT[_disc_cov_rows] = (
    disc_cov_pd.iloc[disc_cov_index[_disc_cov_rows]][DISC_COV_COLS].to_numpy(dtype=float))
print(f"  disc HAS_COV: {DISC_HAS_COV.sum()} / {N_DISC}")

COHORTS = [
    {"label": "10k_disc", "vcf_resolver": lambda chrom: SV_VCF_PATH,
     "n_samples": N_DISC, "sample_ids": DISC_INT, "has_cov": DISC_HAS_COV,
     "cov_mat": DISC_COV_MAT, "cov_cols": DISC_COV_COLS, "n_pcs": N_PCS_DISC,
     "sex_at_birth": DISC_COV_MAT[:, DISC_COV_COLS.index("sex_at_birth")],
     "sample_indices": None},
]

def plot_pval_side_by_side(results_by_cohort, chrom, lead_pos, lead_ref, lead_alt,
                           win_start, win_end, phecode, phecode_name, gene_str,
                           meta_p, out_png):
    """
    results_by_cohort: list of dicts, each with keys:
      label, var_ids, positions, pip, p_s, r2, cs_out,
      is_sv_list, sv_type_list, lead_idx, min_p_idx, n_ym, n_cases, n_pcs,
      min_p_var, min_p, min_pip
    r2 is now computed relative to the min-p variant.
    """
    n_panels = len(results_by_cohort)
    fig = plt.figure(figsize=(13 * n_panels + 1, 13.0))
    gs  = fig.add_gridspec(2, n_panels + 1,
                           width_ratios=[1] * n_panels + [0.03],
                           height_ratios=[7.5, 4.9], hspace=0.04, wspace=0.18)

    cax = fig.add_subplot(gs[:, n_panels])
    sc_for_cbar = None

    for pi, res in enumerate(results_by_cohort):
        ax_pv = fig.add_subplot(gs[0, pi])
        ax_g  = fig.add_subplot(gs[1, pi], sharex=ax_pv)

        positions    = res["positions"]
        p_s          = res["p_s"]
        r2           = res["r2"]
        is_sv_list   = res["is_sv_list"]
        sv_type_list = res["sv_type_list"]
        lead_idx     = res["lead_idx"]
        min_p_idx    = res.get("min_p_idx")
        pip          = res["pip"]
        cs_out       = res["cs_out"]
        var_ids      = res["var_ids"]

        pos_mb      = np.array(positions) / 1e6
        sv_arr      = np.array(is_sv_list, dtype=bool)
        sv_type_arr = np.array(sv_type_list)
        nlp         = -np.log10(np.clip(p_s, 1e-300, 1.0))

        ins_mask = sv_arr & (sv_type_arr == "INS")
        del_mask = sv_arr & (sv_type_arr == "DEL")
        snv_mask = ~sv_arr.copy()

        r2_safe = np.where(np.isfinite(r2), r2, 0.0)
        cmap = plt.cm.viridis

        order = np.argsort(r2_safe[snv_mask])
        sc = ax_pv.scatter(pos_mb[snv_mask][order], nlp[snv_mask][order],
                           s=70, c=r2_safe[snv_mask][order], cmap="viridis",
                           vmin=0, vmax=1, alpha=0.9, edgecolor="none")
        if sc_for_cbar is None:
            sc_for_cbar = sc

        if ins_mask.any():
            ax_pv.scatter(pos_mb[ins_mask], nlp[ins_mask], s=200, marker="^",
                          c=r2_safe[ins_mask], cmap="viridis", vmin=0, vmax=1,
                          alpha=0.9, edgecolor="none")
        if del_mask.any():
            ax_pv.scatter(pos_mb[del_mask], nlp[del_mask], s=200, marker="v",
                          c=r2_safe[del_mask], cmap="viridis", vmin=0, vmax=1,
                          alpha=0.9, edgecolor="none")

        ax_pv.set_ylabel(r"$-\log_{10}$(p)")
        ax_pv.tick_params(labelbottom=False)

        ax_pv.set_title(
            f"10k discovery · {res['sex_scope']} · n={res['n_ym']:,} · cases={res['n_cases']:,}",
            fontsize=FONT, pad=18)

        draw_gene_track(ax_g, chrom, win_start, win_end)
        ax_g.set_xlabel(f"chr{chrom} position (Mb)")

    if sc_for_cbar is not None:
        cbar = fig.colorbar(sc_for_cbar, cax=cax)
        cbar.set_label("r² to min-p variant", fontsize=FONT - 4)
        cbar.ax.tick_params(labelsize=FONT - 4)
        cbar.ax.set_ylim(-COLORBAR_VERTICAL_HEADROOM,
                            1.0 + COLORBAR_VERTICAL_HEADROOM)

    meta_p_str = f"p_meta={meta_p:.2g}" if pd.notna(meta_p) else ""
    fig.suptitle(
        f"chr{chrom}:{lead_pos} {lead_ref}>{lead_alt}"
        f"{(' — ' + gene_str) if gene_str else ''}"
        f" → {phecode} {phecode_name}\n"
        f"Score-test p-values · {meta_p_str}",
        fontsize=FONT, y=1.04)
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"    ✓ saved {out_png}")

print("\n" + "=" * 72)
print("MAIN — SuSiE fine-mapping (10k discovery, 20 PCs, sex aware)")
print("=" * 72)

all_summary   = []
all_cs_detail = []

for ti, t in tasks.iterrows():
    chrom    = int(t.CHR)
    lead_pos = int(t.POS)
    lead_ref = str(t.REF)
    lead_alt = str(t.ALT)
    phecode  = str(t.phecode)
    phecode_name = str(t.phecode_name)
    gene_str = (str(t.get("gene_in_window", ""))
                if pd.notna(t.get("gene_in_window")) else "")
    meta_p   = t.get("p_meta", np.nan)
    sex_scope = phecode_sex_scope(phecode)

    win_start = max(1, lead_pos - WINDOW // 2)
    win_end   = lead_pos + WINDOW // 2

    tag_base = f"{t.CHR}_{t.POS}_{t.REF}_{t.ALT}__{phecode}"
    out_png  = os.path.join(WORKDIR, f"pval_disc_{tag_base}.png")

    print(f"\n{'─' * 72}")
    print(f"TASK {ti+1}/{len(tasks)}: chr{chrom}:{lead_pos} {lead_ref}>{lead_alt}  "
          f"{phecode} {phecode_name}  ({gene_str})")
    print(f"  window: chr{chrom}:{win_start}-{win_end}  sex={sex_scope}")
    print(f"{'─' * 72}")

    cohort_results = []

    for cohort in COHORTS:
        clabel  = cohort["label"]
        n_samp  = cohort["n_samples"]
        samp_id = cohort["sample_ids"]
        has_cov = cohort["has_cov"]
        cov_mat = cohort["cov_mat"]
        cov_cols = cohort["cov_cols"]
        n_pcs   = cohort["n_pcs"]
        sex_at_birth = cohort["sex_at_birth"]
        sample_indices = cohort["sample_indices"]

        tag = f"{tag_base}__{clabel}"
        out_tsv = os.path.join(WORKDIR, f"susie_{tag}.tsv")

        loaded_from_cache = False
        if os.path.exists(out_tsv) and os.path.getsize(out_tsv) > 0:
            try:
                cached_df = pd.read_csv(out_tsv, sep="\t")

                r2_col = ("r2_to_minp" if "r2_to_minp" in cached_df.columns
                          else "r2_to_lead" if "r2_to_lead" in cached_df.columns
                          else None)
                cached_size = (int(cached_df["_cohort_size"].iloc[0])
                               if "_cohort_size" in cached_df.columns else -1)
                cached_pcs = (int(cached_df["_n_pcs"].iloc[0])
                              if "_n_pcs" in cached_df.columns else -1)
                cached_contract = (str(cached_df["_analysis_contract"].iloc[0])
                                   if "_analysis_contract" in cached_df.columns else "")
                cached_sex = (str(cached_df["_sex_scope"].iloc[0])
                              if "_sex_scope" in cached_df.columns else "")
                cached_window_start = (int(cached_df["_window_start"].iloc[0])
                                       if "_window_start" in cached_df.columns else -1)
                cached_window_end = (int(cached_df["_window_end"].iloc[0])
                                     if "_window_end" in cached_df.columns else -1)
                if (r2_col is not None and "lead_variant" in cached_df.columns
                        and cached_size == n_samp and cached_pcs == N_PCS_DISC
                        and cached_contract == ANALYSIS_CONTRACT
                        and cached_sex == sex_scope
                        and cached_window_start == win_start
                        and cached_window_end == win_end):
                    print(f"\n  [{clabel}] ↺ loading cached: {out_tsv}")
                    var_ids      = cached_df["variant_id"].tolist()
                    positions    = cached_df["pos"].tolist()
                    pip          = cached_df["pip"].values
                    p_s          = cached_df["p_score"].values
                    r2           = cached_df[r2_col].values
                    is_sv_list   = cached_df["is_sv"].tolist()
                    sv_type_list = cached_df["sv_type"].fillna("").tolist()

                    cs_out = {}
                    for j, cs_str in enumerate(cached_df["cs"].fillna("")):
                        if cs_str:
                            for cs_name in cs_str.split(","):
                                cs_name = cs_name.strip()
                                if cs_name:
                                    cs_out.setdefault(cs_name, []).append(j)

                    lead_idx = None
                    if cached_df["lead_variant"].any():
                        lead_idx = int(cached_df[cached_df["lead_variant"]].index[0])

                    valid_p = np.where(np.isfinite(p_s) & (p_s > 0), p_s, np.inf)
                    min_p_idx = int(np.argmin(valid_p))
                    min_p     = float(valid_p[min_p_idx])
                    min_p_var = var_ids[min_p_idx]
                    min_pip   = float(pip[min_p_idx])

                    n_ym    = int(cached_df["_n_samples"].iloc[0]) if "_n_samples" in cached_df.columns else 0
                    n_cases = int(cached_df["_n_cases"].iloc[0]) if "_n_cases" in cached_df.columns else 0
                    _cached_mac = int(cached_df["_min_p_mac"].iloc[0]) if "_min_p_mac" in cached_df.columns else 0
                    _cached_method = str(cached_df["_min_p_method"].iloc[0]) if "_min_p_method" in cached_df.columns else ""

                    cohort_results.append({
                        "label": clabel, "var_ids": var_ids, "positions": positions,
                        "pip": pip, "p_s": p_s, "r2": r2, "cs_out": cs_out,
                        "is_sv_list": is_sv_list, "sv_type_list": sv_type_list,
                        "lead_idx": lead_idx, "n_ym": n_ym, "n_cases": n_cases,
                        "n_pcs": n_pcs, "sex_scope": sex_scope,
                        "min_p_idx": min_p_idx, "min_p": min_p,
                        "min_p_var": min_p_var, "min_pip": min_pip,
                        "min_p_mac": _cached_mac,
                        "min_p_method": _cached_method,
                    })
                    loaded_from_cache = True
            except Exception:
                pass

        if loaded_from_cache:
            continue

        print(f"\n  [{clabel}] fine-mapping …")

        vcf_path = cohort["vcf_resolver"](chrom)
        if vcf_path is None or not os.path.exists(vcf_path):
            print(f"    ⚠ no VCF for chr{chrom} in {clabel}"); continue

        y_all = build_phenotype(phecode, samp_id, has_cov, sex_at_birth)
        mask  = ~np.isnan(y_all)
        n_cases = int(np.nansum(y_all)); n_ctrl = int(mask.sum() - n_cases)
        if n_cases < MIN_CASES or n_ctrl < MIN_CASES:
            print(f"    skip: cases={n_cases} controls={n_ctrl}"); continue

        var_ids, X, carrier_matrix, positions, is_sv_list, sv_type_list = \
            fetch_region_cyvcf2(vcf_path, chrom, win_start, win_end, n_samp,
                                sample_indices=sample_indices)
        if X.shape[1] == 0:
            print("    skip: no variants in region"); continue

        Xm = X[mask].astype(float); ym = y_all[mask].astype(float)
        Zm = cov_mat[mask].astype(float) if USE_COVARIATES else None
        sv_arr      = np.array(is_sv_list, dtype=bool)
        sv_type_arr = np.array(sv_type_list)
        n_sv  = int(sv_arr.sum())
        n_ins = int((sv_type_arr == "INS").sum())
        n_del = int((sv_type_arr == "DEL").sum())
        print(f"    variants={Xm.shape[1]}  SVs={n_sv} (INS={n_ins}, DEL={n_del})  "
              f"n={len(ym)}  cases={n_cases}")

        lead_idx = None
        lead_vid = f"chr{chrom}:{lead_pos}:{lead_ref}:{lead_alt}"
        for i, vid in enumerate(var_ids):
            if vid == lead_vid:
                lead_idx = i; break
        if lead_idx is None:
            for i, (p, sv) in enumerate(zip(positions, is_sv_list)):
                if p == lead_pos and not sv:
                    lead_idx = i; break
        if lead_idx is None:
            print(f"    ⚠ lead SNV {lead_vid} not found in VCF — skipping")
            continue

        selected_lead_id = var_ids[lead_idx]
        carrier_counts = np.sum(carrier_matrix[mask], axis=0).astype(int)
        eligible = carrier_counts >= MIN_CARRIERS
        removed = int((~eligible).sum())
        if not eligible[lead_idx]:
            print(f"    skip: lead has {carrier_counts[lead_idx]} carriers; "
                  f"minimum is {MIN_CARRIERS}")
            continue
        Xm = Xm[:, eligible]
        carrier_counts = carrier_counts[eligible]
        var_ids = [value for value, keep in zip(var_ids, eligible) if keep]
        positions = [value for value, keep in zip(positions, eligible) if keep]
        is_sv_list = [value for value, keep in zip(is_sv_list, eligible) if keep]
        sv_type_list = [value for value, keep in zip(sv_type_list, eligible) if keep]
        lead_idx = var_ids.index(selected_lead_id)
        sv_arr = np.array(is_sv_list, dtype=bool)
        sv_type_arr = np.array(sv_type_list)
        n_sv = int(sv_arr.sum())
        n_ins = int((sv_type_arr == "INS").sum())
        n_del = int((sv_type_arr == "DEL").sum())
        print(f"    carrier filter >= {MIN_CARRIERS}: removed={removed}, "
              f"eligible={Xm.shape[1]}")

        if USE_COVARIATES:
            X_susie = residualize(Xm, Zm)
            y_use = residualize(ym, Zm)
        else:
            X_susie = Xm - Xm.mean(axis=0, keepdims=True)
            y_use = ym - ym.mean()
        try:
            pip, cs_out = run_susie(X_susie, y_use, L=L_SUSIE)
        except Exception as exc:
            print(f"    susie failed: {exc}"); continue

        Z_use = Zm if USE_COVARIATES else np.zeros((len(ym), 1))
        beta_s, se_s, p_s = score_test_region(Xm, ym, Z_use)
        nlp = -np.log10(np.clip(p_s, 1e-300, 1.0))

        valid_p = np.where(np.isfinite(p_s) & (p_s > 0), p_s, np.inf)
        min_p_idx = int(np.argmin(valid_p))
        min_p     = float(valid_p[min_p_idx])
        min_p_var = var_ids[min_p_idx]
        min_pip   = float(pip[min_p_idx])

        r2 = compute_r2_to_lead_dosage(Xm, min_p_idx)
        _r2_method = "dosage"
        n_finite = int(np.isfinite(r2).sum())
        print(f"      dosage r² to min-p: {n_finite}/{len(r2)} variants")

        wald_lead = wald_logit_single(Xm[:, lead_idx], ym, Z_use)
        lead_car, lead_mac, lead_af = variant_allele_stats(Xm[:, lead_idx])

        wald_min = wald_logit_single(Xm[:, min_p_idx], ym, Z_use)
        min_car, min_mac, min_af = variant_allele_stats(Xm[:, min_p_idx])

        df = pd.DataFrame({
            "variant_id": var_ids, "chrom": [f"chr{chrom}"] * len(var_ids),
            "pos": positions, "is_sv": is_sv_list, "sv_type": sv_type_list,
            "n_carriers": carrier_counts,
            "pip": pip, "cs": [""] * len(var_ids),
            "beta_score": beta_s, "se_score": se_s, "p_score": p_s,
            "neg_log_p": nlp, "r2_to_minp": r2,
            "r2_method": [_r2_method] * len(var_ids),
        })
        for cs_name, idxs in cs_out.items():
            for j in idxs:
                prev = df.at[j, "cs"]
                df.at[j, "cs"] = f"{prev},{cs_name}" if prev else cs_name
        df["lead_variant"] = False
        df.at[lead_idx, "lead_variant"] = True
        df["minp_variant"] = False
        df.at[min_p_idx, "minp_variant"] = True
        df["_n_samples"] = len(ym)
        df["_cohort_size"] = n_samp
        df["_n_cases"] = n_cases
        df["_n_pcs"] = n_pcs
        df["_sex_scope"] = sex_scope
        df["_analysis_contract"] = ANALYSIS_CONTRACT
        df["_min_carriers"] = MIN_CARRIERS
        df["_window_start"] = win_start
        df["_window_end"] = win_end
        df["_min_p_mac"] = min_mac
        df["_min_p_method"] = wald_min["method"] if wald_min else ""
        df.to_csv(out_tsv, sep="\t", index=False)

        cs_member_idxs = sorted({j for idxs in cs_out.values() for j in idxs})
        for j in cs_member_idxs:
            w = wald_logit_single(Xm[:, j], ym, Z_use)
            all_cs_detail.append({
                "variant_tag": t.variant_tag, "lead_chrom": chrom,
                "lead_pos": lead_pos, "phecode": phecode,
                "phecode_name": phecode_name, "cohort": clabel,
                "cs": df.at[j, "cs"], "variant_id": var_ids[j],
                "pos": positions[j], "is_sv": bool(is_sv_list[j]),
                "sv_type": sv_type_list[j],
                "is_lead": (j == lead_idx),
                "pip": float(pip[j]),
                "r2_to_minp": float(r2[j]) if np.isfinite(r2[j]) else np.nan,
                "beta_score": float(beta_s[j]), "se_score": float(se_s[j]),
                "p_score": float(p_s[j]),
                "test_method": w["method"] if w else "",
                "or_test":     w["or_"]    if w else np.nan,
                "p_test":      w["p"]      if w else np.nan,
                "beta_wald":   w.get("beta", np.nan) if w else np.nan,
                "se_wald":     w.get("se",   np.nan) if w else np.nan,
            })

        n_cs      = len(cs_out)
        cs_sizes  = ";".join(str(len(v)) for v in cs_out.values())
        cs_has_sv = sum(1 for idxs in cs_out.values()
                        if any(is_sv_list[j] for j in idxs))
        lead_pip   = float(pip[lead_idx])
        lead_in_cs = bool(df.at[lead_idx, "cs"] != "")

        print(f"  ─ [{clabel}] lead SNV ─")
        if wald_lead:
            beta_str = (f"beta={wald_lead['beta']:+.4f}"
                        if 'beta' in wald_lead
                        else f"OR={wald_lead['or_']:.4f}(fisher)")
            print(f"    {lead_vid}  carriers={lead_car}  AF={lead_af:.4f}  "
                  f"{beta_str}  p={wald_lead['p']:.2e}  "
                  f"PIP={lead_pip:.3f}  in_CS={lead_in_cs}")
        min_kind = sv_type_list[min_p_idx] if is_sv_list[min_p_idx] else "SNV"
        min_p_short = _short_variant_label(min_p_var, is_sv_list[min_p_idx],
                                           sv_type_list[min_p_idx])
        print(f"  ─ [{clabel}] min-p variant ─")
        print(f"    {min_p_short}  [{min_kind}]  p={min_p:.2e}  "
              f"PIP={min_pip:.3f}  carriers={min_car}  AF={min_af:.4f}")
        if is_sv_list[min_p_idx]:
            print(f"    full ID: {min_p_var}")
        if n_cs:
            print(f"  ─ [{clabel}] {n_cs} credible set(s)  sizes={cs_sizes}  "
                  f"({cs_has_sv} contain ≥1 SV) ─")
            for name in sorted(cs_out):
                for j in sorted(cs_out[name], key=lambda j: -pip[j]):
                    kind = sv_type_list[j] if is_sv_list[j] else "SNV"
                    ld   = (f"r²={r2[j]:.3f}" if np.isfinite(r2[j])
                            else "r²=NA")
                    flag = "  ← LEAD" if j == lead_idx else ""
                    flag += "  ← MIN-P" if j == min_p_idx else ""
                    print(f"      CS {name}  [{kind:3s}] "
                          f"chr{chrom}:{positions[j]:>10}  "
                          f"PIP={pip[j]:.3f}  p={p_s[j]:.2e}  {ld}  "
                          f"{var_ids[j]}{flag}")
        else:
            print(f"  ─ [{clabel}] no credible sets ─")

        all_summary.append({
            "variant_tag": t.variant_tag, "chrom": chrom,
            "lead_pos": lead_pos, "lead_ref": lead_ref, "lead_alt": lead_alt,
            "phecode": phecode, "phecode_name": phecode_name,
            "gene_in_window": gene_str, "cohort": clabel,
            "sex_scope": sex_scope, "analysis_contract": ANALYSIS_CONTRACT,
            "n_samples": int(mask.sum()), "n_cases": n_cases,
            "n_controls": n_ctrl, "n_pcs": n_pcs,
            "n_variants": int(Xm.shape[1]), "n_svs": n_sv,
            "n_ins": n_ins, "n_del": n_del,

            "lead_variant":  lead_vid,
            "lead_pip":      lead_pip,
            "lead_p_score":  float(p_s[lead_idx]),
            "lead_in_cs":    lead_in_cs,
            "lead_carriers": lead_car, "lead_af": lead_af,
            "lead_method":   wald_lead["method"] if wald_lead else "",
            "lead_or":       wald_lead["or_"]    if wald_lead else np.nan,
            "lead_p_wald":   wald_lead["p"]      if wald_lead else np.nan,
            "lead_beta":     wald_lead.get("beta", np.nan) if wald_lead else np.nan,

            "min_p_variant": min_p_var,
            "min_p":         min_p,
            "min_p_pip":     min_pip,
            "min_p_is_sv":   bool(is_sv_list[min_p_idx]),
            "min_p_sv_type": sv_type_list[min_p_idx],
            "min_p_pos":     int(positions[min_p_idx]),
            "min_p_carriers": min_car, "min_p_af": min_af,
            "min_p_or":      wald_min["or_"]  if wald_min else np.nan,
            "min_p_wald":    wald_min["p"]    if wald_min else np.nan,
            "min_p_beta":    wald_min.get("beta", np.nan) if wald_min else np.nan,

            "n_credible_sets": n_cs, "cs_sizes": cs_sizes,
            "n_cs_with_sv":    cs_has_sv,

            "p_meta":        meta_p,
            "beta_meta":     t.get("beta_meta", np.nan),
            "p_disc":        t.get("p_d", np.nan),
            "beta_disc":     t.get("beta_d", np.nan),
        })

        cohort_results.append({
            "label": clabel, "var_ids": var_ids, "positions": positions,
            "pip": pip, "p_s": p_s, "r2": r2, "cs_out": cs_out,
            "is_sv_list": is_sv_list, "sv_type_list": sv_type_list,
            "lead_idx": lead_idx, "n_ym": len(ym), "n_cases": n_cases,
            "n_pcs": n_pcs, "sex_scope": sex_scope,
            "min_p_idx": min_p_idx, "min_p": min_p,
            "min_p_var": min_p_var, "min_pip": min_pip,
            "min_p_mac": min_mac,
            "min_p_method": wald_min["method"] if wald_min else "",
        })

    if cohort_results:
        plot_pval_side_by_side(
            cohort_results, chrom, lead_pos, lead_ref, lead_alt,
            win_start, win_end, phecode, phecode_name, gene_str,
            meta_p, out_png)

print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)

if all_summary:
    summary_df = pd.DataFrame(all_summary)
    sum_path   = os.path.join(WORKDIR, "susie_meta_snv_disc_summary.tsv")
    summary_df.to_csv(sum_path, sep="\t", index=False)
    print(f"✓ Summary → {sum_path}  ({len(summary_df)} rows)")

    show = ["variant_tag", "cohort", "phecode", "phecode_name",
            "gene_in_window", "n_samples", "n_cases",
            "n_variants", "n_svs",
            "lead_variant", "lead_pip", "lead_p_score", "lead_in_cs",
            "min_p_variant", "min_p", "min_p_pip",
            "min_p_is_sv", "min_p_sv_type",
            "n_credible_sets", "cs_sizes", "n_cs_with_sv", "p_meta"]
    show = [c for c in show if c in summary_df.columns]
    print(f"\nKey results (per cohort × phecode):")
    with pd.option_context("display.max_colwidth", 40, "display.width", 320,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(summary_df[show].to_string(index=False))

    print(f"\n{'─' * 72}")
    print("Lowest p-value hit per cohort × phecode:")
    print(f"{'─' * 72}")
    highlight = ["variant_tag", "cohort", "phecode", "phecode_name",
                 "min_p_variant", "min_p", "min_p_pip",
                 "min_p_is_sv", "min_p_sv_type", "min_p_pos",
                 "min_p_carriers", "min_p_af", "min_p_or",
                 "lead_variant", "lead_pip", "p_meta"]
    highlight = [c for c in highlight if c in summary_df.columns]
    with pd.option_context("display.max_colwidth", 40, "display.width", 320,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(summary_df[highlight].to_string(index=False))
else:
    print("  No tasks completed.")

if all_cs_detail:
    cs_df   = pd.DataFrame(all_cs_detail)
    cs_path = os.path.join(WORKDIR, "susie_meta_snv_disc_cs_members.tsv")
    cs_df.to_csv(cs_path, sep="\t", index=False)
    print(f"\n✓ CS members → {cs_path}  ({len(cs_df)} rows)")
    show_cs = ["variant_tag", "cohort", "variant_id", "pos", "is_sv", "sv_type",
               "is_lead", "pip", "r2_to_minp",
               "p_score", "or_test", "p_test", "test_method", "cs"]
    show_cs = [c for c in show_cs if c in cs_df.columns]
    with pd.option_context("display.max_colwidth", 40, "display.width", 300,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(cs_df[show_cs].to_string(index=False))

print("\n✓ Done — all outputs in", WORKDIR)


In [ ]:
import os, glob, gzip, shutil, urllib.request
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

WORKDIR   = "susie_finemap_meta"
COV_TAG   = "cov20"
PIP_THRESH = 0.01
WINDOW     = 500_000

COLOC_TSV = "AOU Phase 1 Tables (current) - MAGE and GWAS coloc (2).tsv"
GTF_PATH  = "hg38.refGene.gtf"
GTF_CACHE = "hg38.refGene.exons.parquet"
GTF_URL   = ("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/"
             "bigZips/genes/hg38.refGene.gtf.gz")

FONT = 26
plt.rcParams.update({
    "font.size": FONT, "axes.titlesize": FONT, "axes.labelsize": FONT,
    "xtick.labelsize": FONT, "ytick.labelsize": FONT,
    "legend.fontsize": FONT, "figure.titlesize": FONT,
})

def _safe_float(df, idx, col):
    """Return float value from df.loc[idx, col], or np.nan if missing/non-numeric."""
    if col not in df.columns:
        return np.nan
    v = df.loc[idx, col]
    try:
        fv = float(v)
        return fv if pd.notna(fv) else np.nan
    except (ValueError, TypeError):
        return np.nan

def _ensure_gtf():
    if os.path.exists(GTF_CACHE):
        return pl.read_parquet(GTF_CACHE)
    if not os.path.exists(GTF_PATH):
        print(f"Downloading {GTF_URL} → {GTF_PATH} …")
        gz = GTF_PATH + ".gz"
        urllib.request.urlretrieve(GTF_URL, gz)
        with gzip.open(gz, "rb") as fi, open(GTF_PATH, "wb") as fo:
            shutil.copyfileobj(fi, fo)
        os.remove(gz)
    print(f"Parsing {GTF_PATH} (one-time) …")
    df = pl.read_csv(
        GTF_PATH, separator="\t", has_header=False, comment_prefix="#",
        new_columns=["chrom","source","feature","start","end","score",
                     "strand","frame","attribute"],
        schema_overrides={"chrom": pl.Utf8, "start": pl.Int64, "end": pl.Int64,
                          "strand": pl.Utf8, "attribute": pl.Utf8,
                          "score": pl.Utf8, "frame": pl.Utf8},
    ).filter(pl.col("feature") == "exon")
    df = df.with_columns([
        pl.col("attribute").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_name"),
    ]).select(["chrom", "start", "end", "gene_name"])
    df.write_parquet(GTF_CACHE)
    return df

GTF_EXONS = _ensure_gtf()
print(f"GTF exons loaded: {GTF_EXONS.height:,} rows")

def variant_in_exon(chrom_str, pos):
    """Return list of gene names whose exons overlap this 1-based position."""
    hits = GTF_EXONS.filter(
        (pl.col("chrom") == chrom_str)
        & (pl.col("start") <= pos)
        & (pl.col("end")   >= pos)
    )
    if hits.height == 0:
        return []
    return hits["gene_name"].unique().to_list()

sum_path = os.path.join(WORKDIR, f"susie_meta_snv_summary_{COV_TAG}.tsv")
if not os.path.exists(sum_path):
    raise FileNotFoundError(
        f"{sum_path} not found — run the main fine-mapping pipeline first.")
summary = pd.read_csv(sum_path, sep="\t")
print(f"Summary loaded: {len(summary)} fine-mapped windows")

region_files = sorted(glob.glob(os.path.join(WORKDIR, f"susie_*__{COV_TAG}.tsv")))
region_files = [f for f in region_files
                if not os.path.basename(f).startswith("susie_meta_snv")]
print(f"Per-region TSVs found: {len(region_files)}")

_sample_cols = set()
if region_files:
    _sample_cols = set(pd.read_csv(region_files[0], sep="\t", nrows=0).columns)
    print(f"Region TSV columns: {sorted(_sample_cols)}")

STAT_COLS = [c for c in [
    "beta_score", "se_score", "beta_wald", "se_wald",
    "or_test", "p_test", "test_method", "r2_to_lead",
    "af",
] if c in _sample_cols]
print(f"Stat columns available: {STAT_COLS if STAT_COLS else '(none — will compute AF from dosage if possible)'}")

exon_rows = []

for _, row in summary.iterrows():
    chrom_int = int(row["chrom"])
    lead_pos  = int(row["pos"])
    phecode   = str(row["phecode"])
    tag       = f"{chrom_int}_{lead_pos}_{row['ref']}_{row['alt']}__{phecode}__{COV_TAG}"
    reg_path  = os.path.join(WORKDIR, f"susie_{tag}.tsv")

    if not os.path.exists(reg_path):
        continue
    reg = pd.read_csv(reg_path, sep="\t")

    chrom_str = f"chr{chrom_int}"

    for nc in ["p_score", "pip", "beta_score", "se_score",
               "beta_wald", "se_wald", "or_test", "p_test",
               "r2_to_lead", "af"]:
        if nc in reg.columns:
            reg[nc] = pd.to_numeric(reg[nc], errors="coerce")

    valid_p = reg.dropna(subset=["p_score"])
    if len(valid_p):
        bp_idx   = valid_p["p_score"].idxmin()
        bp_vid   = valid_p.loc[bp_idx, "variant_id"]
        bp_pos   = int(valid_p.loc[bp_idx, "pos"])
        bp_val   = float(valid_p.loc[bp_idx, "p_score"])
        bp_pip   = _safe_float(valid_p, bp_idx, "pip")
        bp_is_sv = bool(valid_p.loc[bp_idx, "is_sv"])
        bp_exon  = variant_in_exon(chrom_str, bp_pos)

        bp_beta_score = _safe_float(valid_p, bp_idx, "beta_score")
        bp_se_score   = _safe_float(valid_p, bp_idx, "se_score")
        bp_beta_wald  = _safe_float(valid_p, bp_idx, "beta_wald")
        bp_se_wald    = _safe_float(valid_p, bp_idx, "se_wald")
        bp_or_test    = _safe_float(valid_p, bp_idx, "or_test")
        bp_p_test     = _safe_float(valid_p, bp_idx, "p_test")
        bp_af         = _safe_float(valid_p, bp_idx, "af")
        bp_r2_to_lead = _safe_float(valid_p, bp_idx, "r2_to_lead")
    else:
        bp_vid, bp_pos, bp_val, bp_pip = "", None, None, np.nan
        bp_is_sv, bp_exon = False, []
        bp_beta_score = bp_se_score = bp_beta_wald = bp_se_wald = np.nan
        bp_or_test = bp_p_test = bp_af = bp_r2_to_lead = np.nan

    valid_pip = reg.dropna(subset=["pip"])
    if len(valid_pip):
        hp_idx   = valid_pip["pip"].idxmax()
        hp_vid   = valid_pip.loc[hp_idx, "variant_id"]
        hp_pos   = int(valid_pip.loc[hp_idx, "pos"])
        hp_pip   = float(valid_pip.loc[hp_idx, "pip"])
        hp_pval  = _safe_float(valid_pip, hp_idx, "p_score")
        hp_is_sv = bool(valid_pip.loc[hp_idx, "is_sv"])
        hp_exon  = variant_in_exon(chrom_str, hp_pos)

        hp_beta_score = _safe_float(valid_pip, hp_idx, "beta_score")
        hp_se_score   = _safe_float(valid_pip, hp_idx, "se_score")
        hp_beta_wald  = _safe_float(valid_pip, hp_idx, "beta_wald")
        hp_se_wald    = _safe_float(valid_pip, hp_idx, "se_wald")
        hp_or_test    = _safe_float(valid_pip, hp_idx, "or_test")
        hp_p_test     = _safe_float(valid_pip, hp_idx, "p_test")
        hp_af         = _safe_float(valid_pip, hp_idx, "af")
        hp_r2_to_lead = _safe_float(valid_pip, hp_idx, "r2_to_lead")
    else:
        hp_vid, hp_pos, hp_pip, hp_pval = "", None, np.nan, np.nan
        hp_is_sv, hp_exon = False, []
        hp_beta_score = hp_se_score = hp_beta_wald = hp_se_wald = np.nan
        hp_or_test = hp_p_test = hp_af = hp_r2_to_lead = np.nan

    hip = reg[reg["pip"] > PIP_THRESH].copy()
    pip_in_exon_genes = set()
    pip_in_exon_vids  = []
    for _, v in hip.iterrows():
        genes = variant_in_exon(chrom_str, int(v["pos"]))
        if genes:
            pip_in_exon_genes.update(genes)
            pip_in_exon_vids.append(v["variant_id"])

    any_exonic = bool(bp_exon) or bool(hp_exon) or bool(pip_in_exon_genes)

    vtag = str(row.get("variant_tag",
                        f"{chrom_int}_{lead_pos}_{row['ref']}_{row['alt']}"))

    exon_rows.append({
        "variant_tag":          vtag,
        "chrom":                chrom_int,
        "pos":                  lead_pos,
        "ref":                  row["ref"],
        "alt":                  row["alt"],
        "phecode":              phecode,
        "phecode_name":         row.get("phecode_name", ""),
        "gene_in_window":       row.get("gene_in_window", ""),
        "n_cases":              int(row.get("n_cases", 0)),
        "lead_pip":             float(row.get("lead_pip", np.nan)),
        "lead_p_wald":          float(row.get("lead_p_wald", np.nan)),

        "best_p_variant":       bp_vid,
        "best_p_pos":           bp_pos,
        "best_p_value":         bp_val,
        "best_p_pip":           bp_pip,
        "best_p_is_sv":         bp_is_sv,
        "best_p_in_exon":       bool(bp_exon),
        "best_p_exon_genes":    ";".join(sorted(set(bp_exon))) if bp_exon else "",
        "best_p_af":            bp_af,
        "best_p_beta_score":    bp_beta_score,
        "best_p_se_score":      bp_se_score,
        "best_p_beta_wald":     bp_beta_wald,
        "best_p_se_wald":       bp_se_wald,
        "best_p_or":            bp_or_test,
        "best_p_p_wald":        bp_p_test,
        "best_p_r2_to_lead":    bp_r2_to_lead,

        "top_pip_variant":      hp_vid,
        "top_pip_pos":          hp_pos,
        "top_pip_value":        hp_pip,
        "top_pip_pval":         hp_pval,
        "top_pip_is_sv":        hp_is_sv,
        "top_pip_in_exon":      bool(hp_exon),
        "top_pip_exon_genes":   ";".join(sorted(set(hp_exon))) if hp_exon else "",
        "top_pip_af":           hp_af,
        "top_pip_beta_score":   hp_beta_score,
        "top_pip_se_score":     hp_se_score,
        "top_pip_beta_wald":    hp_beta_wald,
        "top_pip_se_wald":      hp_se_wald,
        "top_pip_or":           hp_or_test,
        "top_pip_p_wald":       hp_p_test,
        "top_pip_r2_to_lead":   hp_r2_to_lead,

        "n_pip_above_thresh":   len(hip),
        "n_pip_in_exon":        len(pip_in_exon_vids),
        "pip_exon_genes":       ";".join(sorted(pip_in_exon_genes)) if pip_in_exon_genes else "",
        "pip_exon_variants":    ";".join(pip_in_exon_vids) if pip_in_exon_vids else "",
        "any_exonic_signal":    any_exonic,
    })

exon_df = pd.DataFrame(exon_rows)

vtag_info = (exon_df.drop_duplicates(subset=["variant_tag"])
             [["variant_tag", "chrom", "pos", "ref", "alt", "gene_in_window"]]
             .sort_values(["chrom", "pos"]).reset_index(drop=True))

lead_exon_genes = []
for _, r in vtag_info.iterrows():
    genes = variant_in_exon(f"chr{int(r['chrom'])}", int(r['pos']))
    lead_exon_genes.append(";".join(sorted(set(genes))) if genes else "")
vtag_info["lead_in_exon"]   = [bool(g) for g in lead_exon_genes]
vtag_info["lead_exon_gene"] = lead_exon_genes

dup_flags = [""] * len(vtag_info)
locus_id  = [0]  * len(vtag_info)
cur_locus = 0
for i in range(len(vtag_info)):
    if i == 0:
        locus_id[i] = cur_locus
        continue
    same_chrom = vtag_info.iloc[i]["chrom"] == vtag_info.iloc[i - 1]["chrom"]
    close      = abs(vtag_info.iloc[i]["pos"] - vtag_info.iloc[i - 1]["pos"]) <= WINDOW
    if same_chrom and close:
        locus_id[i] = locus_id[i - 1]
        dist = abs(vtag_info.iloc[i]["pos"] - vtag_info.iloc[i - 1]["pos"])
        dup_flags[i]     = f"← overlap w/ above ({dist:,} bp)"
        if not dup_flags[i - 1]:
            dup_flags[i - 1] = "← overlap w/ below"
    else:
        cur_locus += 1
        locus_id[i] = cur_locus
vtag_info["locus_id"] = locus_id
vtag_info["overlap"]  = dup_flags
n_unique_loci = vtag_info["locus_id"].nunique()

print(f"\nVariant tags: {len(vtag_info)}  →  {n_unique_loci} independent loci "
      f"(WINDOW = ±{WINDOW//1000} kb)")
print(f"{'─'*105}")
for _, r in vtag_info.iterrows():
    flag = f"  {r['overlap']}" if r["overlap"] else ""
    exon_note = f"  [EXON: {r['lead_exon_gene']}]" if r["lead_in_exon"] else ""
    print(f"  L{r['locus_id']:>2d}  chr{r['chrom']}:{r['pos']:>11,}  "
          f"{r['ref']}>{r['alt']}  {str(r['gene_in_window']):<30s}"
          f"{exon_note}{flag}")
print(f"{'─'*105}")

_vtag_idx = vtag_info.set_index("variant_tag")
exon_df["locus_id"]       = exon_df["variant_tag"].map(_vtag_idx["locus_id"])
exon_df["lead_in_exon"]   = exon_df["variant_tag"].map(_vtag_idx["lead_in_exon"])
exon_df["lead_exon_gene"] = exon_df["variant_tag"].map(_vtag_idx["lead_exon_gene"])

n_total        = len(exon_df)
n_bestp_exon   = int(exon_df["best_p_in_exon"].sum())
n_toppip_exon  = int(exon_df["top_pip_in_exon"].sum())
n_anypip_exon  = int((exon_df["n_pip_in_exon"] > 0).sum())
n_any_exon     = int(exon_df["any_exonic_signal"].sum())

print("\n" + "═" * 72)
print("  EXON OVERLAP SUMMARY — meta-significant fine-mapped windows")
print("═" * 72)
print(f"  Total windows (variant × phecode)     : {n_total}")
print(f"  Best p-value variant in exon           : {n_bestp_exon:>4}  "
      f"({100*n_bestp_exon/n_total:.1f}%)")
print(f"  Highest PIP variant in exon            : {n_toppip_exon:>4}  "
      f"({100*n_toppip_exon/n_total:.1f}%)")
print(f"  Any PIP > {PIP_THRESH} variant in exon        : {n_anypip_exon:>4}  "
      f"({100*n_anypip_exon/n_total:.1f}%)")
print(f"  Any criterion (any exonic signal)      : {n_any_exon:>4}  "
      f"({100*n_any_exon/n_total:.1f}%)")
print("═" * 72)

with open(COLOC_TSV) as fh:
    for skip_n, line in enumerate(fh):
        if line.startswith("evidence_types"):
            break
coloc = pd.read_csv(COLOC_TSV, sep="\t", skiprows=skip_n, dtype=str)
coloc.columns = coloc.columns.str.strip()
coloc["max_IHS_position"] = pd.to_numeric(
    coloc["max_IHS_position"], errors="coerce").astype("Int64")
coloc["chrom_int"] = (coloc["chrom"].str.replace("chr", "", regex=False)
                      .apply(pd.to_numeric, errors="coerce").astype("Int64"))
print(f"\nColoc catalog loaded: {len(coloc)} loci")

loci = (exon_df.drop_duplicates(subset=["variant_tag"])
        .reset_index(drop=True).copy())

loci_pheno = (exon_df.groupby(["variant_tag"])
              .agg(
                  phecodes       = ("phecode", lambda x: ";".join(sorted(set(x)))),
                  phecode_names  = ("phecode_name",
                                    lambda x: ";".join(sorted(set(str(v) for v in x)))),
                  n_phecodes     = ("phecode", "nunique"),
                  max_lead_pip   = ("lead_pip", "max"),
                  any_exonic     = ("any_exonic_signal", "any"),
                  best_p_in_exon = ("best_p_in_exon", "any"),
                  top_pip_in_exon = ("top_pip_in_exon", "any"),
                  any_pip_exon   = ("n_pip_in_exon", lambda x: (x > 0).any()),
                  all_exon_genes = ("pip_exon_genes",
                                    lambda x: ";".join(sorted(
                                        set(g for v in x if v
                                            for g in str(v).split(";") if g)))),
              ).reset_index())

_keep_cols = [
    "variant_tag", "chrom", "pos", "ref", "alt",
    "gene_in_window", "locus_id",
    "lead_in_exon", "lead_exon_gene",

    "best_p_variant", "best_p_pos", "best_p_value",
    "best_p_pip", "best_p_is_sv", "best_p_exon_genes",
    "best_p_af", "best_p_beta_score", "best_p_se_score",
    "best_p_beta_wald", "best_p_se_wald", "best_p_or",
    "best_p_p_wald", "best_p_r2_to_lead",

    "top_pip_variant", "top_pip_pos", "top_pip_value",
    "top_pip_pval", "top_pip_is_sv", "top_pip_exon_genes",
    "top_pip_af", "top_pip_beta_score", "top_pip_se_score",
    "top_pip_beta_wald", "top_pip_se_wald", "top_pip_or",
    "top_pip_p_wald", "top_pip_r2_to_lead",
]
_keep_cols = [c for c in _keep_cols if c in loci.columns]
loci = loci[_keep_cols].merge(loci_pheno, on=["variant_tag"], how="left")

loci = loci.merge(
    coloc[["chrom_int", "max_IHS_position", "evidence_types",
           "n_evidence_sources", "has_linked_SV",
           "MAGE_best_egene_symbol", "MAGE_best_pip", "MAGE_n_linked_egenes",
           "SQTL_best_sgene_symbol", "SQTL_best_pip", "SQTL_n_linked_sgenes",
           "GWAS_best_trait", "GWAS_best_pval", "GWAS_best_r2",
           "GWAS_n_unique_traits", "GWAS_linked_traits"]],
    left_on=["chrom", "pos"],
    right_on=["chrom_int", "max_IHS_position"],
    how="left",
    indicator=True,
)
loci["in_coloc_catalog"] = loci["_merge"] == "both"
loci.drop(columns=["_merge", "chrom_int", "max_IHS_position"],
          inplace=True, errors="ignore")

_overlap_map = vtag_info.set_index("variant_tag")["overlap"].to_dict()
loci["overlap_note"] = loci["variant_tag"].map(_overlap_map).fillna("")

def _classify(row):
    if not row["in_coloc_catalog"]:
        return "novel (not in IHS coloc catalog)"
    ev = str(row.get("evidence_types", ""))
    parts = []
    if "eQTL" in ev: parts.append("eQTL")
    if "sQTL" in ev: parts.append("sQTL")
    if "GWAS" in ev: parts.append("GWAS")
    return " + ".join(parts) if parts else "in catalog (no specific evidence)"

loci["coloc_class"] = loci.apply(_classify, axis=1)

for prefix in ["best_p", "top_pip"]:
    or_col   = f"{prefix}_or"
    beta_col = f"{prefix}_beta_wald"

    if or_col in loci.columns and beta_col in loci.columns:
        mask_need = loci[or_col].isna() & loci[beta_col].notna()
        if mask_need.any():
            loci.loc[mask_need, or_col] = np.exp(loci.loc[mask_need, beta_col])

    beta_s_col = f"{prefix}_beta_score"
    if or_col in loci.columns and beta_s_col in loci.columns:
        mask_need2 = loci[or_col].isna() & loci[beta_s_col].notna()
        if mask_need2.any():
            loci.loc[mask_need2, or_col] = np.exp(loci.loc[mask_need2, beta_s_col])

n_loci        = len(loci)
n_indep_loci  = loci["locus_id"].nunique()
n_in_catalog  = int(loci["in_coloc_catalog"].sum())
n_novel       = n_loci - n_in_catalog
n_exonic      = int(loci["any_exonic"].sum())
n_exonic_cat  = int((loci["any_exonic"] & loci["in_coloc_catalog"]).sum())
n_exonic_nov  = int((loci["any_exonic"] & ~loci["in_coloc_catalog"]).sum())

print("\n" + "═" * 72)
print("  CROSS-REFERENCE: fine-mapped windows vs MAGE/GWAS coloc catalog")
print("═" * 72)
print(f"  Unique variant_tags                : {n_loci}")
print(f"  Independent loci (after overlap)   : {n_indep_loci}")
print(f"  Matched to IHS coloc catalog       : {n_in_catalog}")
print(f"  Not in coloc catalog (novel)       : {n_novel}")
print(f"  With exonic signal — total         : {n_exonic}")
print(f"    … and in coloc catalog           : {n_exonic_cat}")
print(f"    … and novel                      : {n_exonic_nov}")
print("═" * 72)

print("\n  Evidence class breakdown:")
class_counts = loci["coloc_class"].value_counts()
for cls, cnt in class_counts.items():
    print(f"    {cls:<45s} : {cnt}")

print(f"\n{'─'*72}")
print("  TABLE A — All fine-mapped loci with coloc catalog annotation")
print(f"{'─'*72}")
tbl_cols_a = [
    "locus_id", "variant_tag", "chrom", "pos", "gene_in_window",
    "n_phecodes", "max_lead_pip",
    "lead_in_exon", "lead_exon_gene",
    "any_exonic", "all_exon_genes",
    "in_coloc_catalog", "coloc_class",
    "evidence_types", "MAGE_best_egene_symbol", "GWAS_best_trait",
    "overlap_note",
]
tbl_cols_a = [c for c in tbl_cols_a if c in loci.columns]
with pd.option_context("display.max_colwidth", 42, "display.width", 320,
                       "display.float_format",
                       lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
    print(loci.sort_values(["locus_id", "pos"])[tbl_cols_a]
          .to_string(index=False))

b1 = loci[loci["top_pip_in_exon"]].copy()
if len(b1):
    print(f"\n{'─'*72}")
    print(f"  TABLE B1 — Highest-PIP variant in exon  ({len(b1)} loci)")
    print(f"{'─'*72}")
    tbl_b1 = [
        "locus_id", "variant_tag", "gene_in_window",
        "top_pip_variant", "top_pip_pos", "top_pip_value",
        "top_pip_pval", "top_pip_is_sv", "top_pip_exon_genes",
        "top_pip_af", "top_pip_beta_score", "top_pip_se_score",
        "top_pip_beta_wald", "top_pip_se_wald", "top_pip_or",
        "top_pip_p_wald", "top_pip_r2_to_lead",
        "n_phecodes", "phecodes",
        "coloc_class", "evidence_types",
        "MAGE_best_egene_symbol", "SQTL_best_sgene_symbol",
        "GWAS_best_trait", "GWAS_best_pval",
    ]
    tbl_b1 = [c for c in tbl_b1 if c in b1.columns]
    with pd.option_context("display.max_colwidth", 40, "display.width", 400,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(b1.sort_values("locus_id")[tbl_b1].to_string(index=False))
else:
    print(f"\n  TABLE B1 — (no windows had the highest-PIP variant in an exon)")

b2 = loci[loci["best_p_in_exon"]].copy()
if len(b2):
    print(f"\n{'─'*72}")
    print(f"  TABLE B2 — Best p-value variant in exon  ({len(b2)} loci)")
    print(f"{'─'*72}")
    tbl_b2 = [
        "locus_id", "variant_tag", "gene_in_window",
        "best_p_variant", "best_p_pos", "best_p_value",
        "best_p_pip", "best_p_is_sv", "best_p_exon_genes",
        "best_p_af", "best_p_beta_score", "best_p_se_score",
        "best_p_beta_wald", "best_p_se_wald", "best_p_or",
        "best_p_p_wald", "best_p_r2_to_lead",
        "n_phecodes", "phecodes",
        "coloc_class", "evidence_types",
        "MAGE_best_egene_symbol", "SQTL_best_sgene_symbol",
        "GWAS_best_trait", "GWAS_best_pval",
    ]
    tbl_b2 = [c for c in tbl_b2 if c in b2.columns]
    with pd.option_context("display.max_colwidth", 40, "display.width", 400,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(b2.sort_values("locus_id")[tbl_b2].to_string(index=False))
else:
    print(f"\n  TABLE B2 — (no windows had the best p-value variant in an exon)")

print(f"\n{'─'*72}")
print(f"  TABLE B3 — Top-PIP & best-p variant effect-size summary (all {n_loci} loci)")
print(f"{'─'*72}")
tbl_b3 = [
    "locus_id", "variant_tag", "gene_in_window",

    "top_pip_variant", "top_pip_value",
    "top_pip_af", "top_pip_beta_score", "top_pip_or",
    "top_pip_pval", "top_pip_is_sv",

    "best_p_variant", "best_p_pip",
    "best_p_af", "best_p_beta_score", "best_p_or",
    "best_p_value", "best_p_is_sv",

    "n_phecodes", "coloc_class",
]
tbl_b3 = [c for c in tbl_b3 if c in loci.columns]
with pd.option_context("display.max_colwidth", 36, "display.width", 420,
                       "display.float_format",
                       lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
    print(loci.sort_values("locus_id")[tbl_b3].to_string(index=False))

novel_loci = loci[~loci["in_coloc_catalog"]].copy()
if len(novel_loci):
    print(f"\n{'─'*72}")
    print(f"  TABLE C — Novel loci (meta-significant, NOT in IHS coloc catalog)"
          f"  ({len(novel_loci)})")
    print(f"{'─'*72}")
    tbl_cols_c = [
        "locus_id", "variant_tag", "chrom", "pos", "ref", "alt",
        "gene_in_window",
        "n_phecodes", "phecodes", "max_lead_pip",
        "lead_in_exon", "lead_exon_gene",
        "any_exonic", "all_exon_genes",

        "top_pip_variant", "top_pip_value",
        "top_pip_af", "top_pip_beta_score", "top_pip_or",
        "best_p_variant", "best_p_value",
        "best_p_af", "best_p_beta_score", "best_p_or",
    ]
    tbl_cols_c = [c for c in tbl_cols_c if c in novel_loci.columns]
    with pd.option_context("display.max_colwidth", 42, "display.width", 420,
                           "display.float_format",
                           lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(novel_loci.sort_values("locus_id")[tbl_cols_c]
              .to_string(index=False))
else:
    print("\n  (All fine-mapped loci matched the coloc catalog — none novel.)")

out_path = os.path.join(WORKDIR, f"exon_coloc_crossref_{COV_TAG}.tsv")
loci.to_csv(out_path, sep="\t", index=False)
print(f"\n✓ Full cross-reference table → {out_path}  ({len(loci)} loci)")

fig, axes = plt.subplots(1, 3, figsize=(28, 8))

labels_a = ["Best p\nin exon", "Top PIP\nin exon",
            f"Any PIP>{PIP_THRESH}\nin exon", "Any\nexonic"]
vals_a   = [
    int(loci["best_p_in_exon"].sum()),
    int(loci["top_pip_in_exon"].sum()),
    int(loci["any_pip_exon"].sum()),
    int(loci["any_exonic"].sum()),
]
colours_a = ["#4C72B0", "#C44E52", "#DD8452", "#55A868"]
bars = axes[0].bar(labels_a, vals_a, color=colours_a, edgecolor="black", lw=1.2)
for b, v in zip(bars, vals_a):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3,
                 str(v), ha="center", va="bottom", fontsize=FONT)
axes[0].set_ylabel("Number of loci")
axes[0].set_title("Exonic overlap in\nmeta-significant windows")
axes[0].set_ylim(0, max(vals_a) * 1.25 if max(vals_a) > 0 else 5)

class_order = sorted(class_counts.index, key=lambda x: -class_counts[x])
vals_b   = [class_counts[c] for c in class_order]
short    = [c.replace("novel (not in IHS coloc catalog)", "Novel")
              .replace("in catalog (no specific evidence)", "Catalog (other)")
            for c in class_order]
colours_b = plt.cm.Set2.colors
bars = axes[1].barh(range(len(vals_b)), vals_b,
                    color=[colours_b[i % len(colours_b)]
                           for i in range(len(vals_b))],
                    edgecolor="black", lw=1.2)
for b, v in zip(bars, vals_b):
    axes[1].text(b.get_width() + 0.2, b.get_y() + b.get_height() / 2,
                 str(v), ha="left", va="center", fontsize=FONT)
axes[1].set_yticks(range(len(short)))
axes[1].set_yticklabels(short, fontsize=FONT - 4)
axes[1].set_xlabel("Number of loci")
axes[1].set_title("Coloc catalog\nevidence class")
axes[1].set_xlim(0, max(vals_b) * 1.3 if max(vals_b) > 0 else 5)
axes[1].invert_yaxis()

or_vals = loci["top_pip_or"].dropna()
if len(or_vals) > 0:
    log_or = np.log(or_vals.replace(0, np.nan).dropna())
    if len(log_or) > 1:
        axes[2].hist(log_or, bins=max(8, len(log_or)//3), color="#8172B2",
                     edgecolor="black", lw=1.2, alpha=0.85)
        axes[2].axvline(0, color="black", ls="--", lw=1.5, alpha=0.5)
        axes[2].set_xlabel("ln(OR) of top-PIP variant")
        axes[2].set_ylabel("Count")
        axes[2].set_title("Effect size distribution\n(top-PIP variants)")
        med_or = float(np.median(or_vals))
        axes[2].text(0.97, 0.95,
                     f"n={len(or_vals)}\nmedian OR={med_or:.2f}",
                     transform=axes[2].transAxes, ha="right", va="top",
                     fontsize=FONT - 2,
                     bbox=dict(boxstyle="round,pad=0.3", fc="white",
                               ec="gray", alpha=0.8))
    else:
        axes[2].text(0.5, 0.5, f"Only {len(log_or)} OR value(s)\navailable",
                     ha="center", va="center", fontsize=FONT,
                     transform=axes[2].transAxes)
        axes[2].set_title("Effect size distribution\n(top-PIP variants)")
else:
    axes[2].text(0.5, 0.5, "No OR data\navailable",
                 ha="center", va="center", fontsize=FONT,
                 transform=axes[2].transAxes)
    axes[2].set_title("Effect size distribution\n(top-PIP variants)")

fig.tight_layout()
fig_path = os.path.join(WORKDIR, f"exon_coloc_summary_{COV_TAG}.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Figure → {fig_path}")


In [ ]:
import os, sys, subprocess, urllib.request, gzip, shutil
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

WORKDIR = "susie_finemap_meta"
os.makedirs(WORKDIR, exist_ok=True)

META_TSV = "AOU Phase 1 Tables (current) - meta-analysis results.tsv"
if not os.path.exists(META_TSV):
    META_TSV = "AOU Phase 1 Tables (current) - meta-analysis results.tsv"

DISC_WORKDIR     = "phewas_sv_v5"
DISC_COV         = f"{DISC_WORKDIR}/covariates_base.tsv"
DISC_PCA         = f"{DISC_WORKDIR}/pca_scores_afr.tsv.bgz"
DISC_PHE         = f"{DISC_WORKDIR}/aou_phecodeX_counts.tsv"
FLAGGED_IDS_PATH = f"{DISC_WORKDIR}/relatedness_flagged_in_cohort.tsv"
SV_VCF           = ("SV_10k/AoU12k.vs.AoU-HPRC-v1.1b.AF_gt_pt005."
                    "25kb_windows.0d0e0eb4.kage.glimpse.collisionless.merged.vcf.gz")

HARDMASK_PATH    = "hardmask.hg38.v4.over99.bed"
GTF_PATH         = "hg38.refGene.gtf"
GTF_CACHE        = "hg38.refGene.exons.parquet"
GTF_URL          = ("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/"
                    "bigZips/genes/hg38.refGene.gtf.gz")

WINDOW           = 500_000
L_SUSIE          = 10
MIN_MAC          = 5
MAX_MISS         = 0.05
MIN_CASES        = 50
MIN_PHECODE_COUNT = 2
SV_LEN_DIFF      = 50

USE_COVARIATES   = True
NEW_N_PCS        = 20
MAX_TASKS        = None

FONT = 34
plt.rcParams.update({
    "font.size": FONT, "axes.titlesize": FONT, "axes.labelsize": FONT,
    "xtick.labelsize": FONT, "ytick.labelsize": FONT, "legend.fontsize": FONT,
    "figure.titlesize": FONT,
})

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                   check=True)

try:
    from cyvcf2 import VCF as CyVCF
except ImportError:
    _pip("cyvcf2"); from cyvcf2 import VCF as CyVCF

from scripts.susie_rscript import ensure_susie_r, run_susie
ensure_susie_r()

with open(META_TSV) as fh:
    header_row = None
    for i, line in enumerate(fh):
        if "variant_tag" in line.split("\t")[0]:
            header_row = i
            break
if header_row is None:
    raise RuntimeError(f"No 'variant_tag' header row in {META_TSV}")

print(f"Reading meta-analysis results from {META_TSV}  (header at line {header_row+1})")
raw = pd.read_csv(META_TSV, sep="\t", skiprows=header_row, dtype=str)
raw.columns = raw.columns.str.strip()

raw = raw.dropna(subset=["variant_tag", "phecode", "meta_significant"])
raw = raw[raw["meta_significant"].str.strip().str.upper() == "TRUE"].copy()
print(f"  meta_significant rows: {len(raw)}")

def _parse_snv_tag(tag):
    parts = str(tag).strip().split("_", 3)
    if len(parts) < 4:
        return None, None, None, None
    try:
        return int(parts[0]), int(parts[1]), parts[2], parts[3]
    except Exception:
        return None, None, None, None

raw[["CHR", "POS", "REF", "ALT"]] = raw["variant_tag"].apply(
    lambda t: pd.Series(_parse_snv_tag(t)))
raw = raw.dropna(subset=["CHR", "POS"]).copy()
raw["CHR"] = raw["CHR"].astype(int)
raw["POS"] = raw["POS"].astype(int)
raw["phecode"]      = raw["phecode"].astype(str).str.strip()
raw["phecode_name"] = (raw["phecode_string"].astype(str).str.strip()
                       if "phecode_string" in raw.columns else raw["phecode"])

for c in ("beta_d","se_d","p_d","n_cases_d","n_controls_d","mac_disc",
          "beta_r","se_r","p_r","n_cases_r","n_controls_r","mac_rep",
          "beta_meta","se_meta","p_meta"):
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

tasks = (raw.drop_duplicates(subset=["variant_tag", "phecode"], keep="first")
            .reset_index(drop=True))
if MAX_TASKS is not None:
    tasks = tasks.head(MAX_TASKS).reset_index(drop=True)
print(f"Tasks to fine-map: {len(tasks)}  "
      f"(covariates={'on' if USE_COVARIATES else 'OFF'})")

_bed = pl.scan_csv(
    HARDMASK_PATH, separator="\t", has_header=False,
    new_columns=["chrom", "start", "end"],
    schema_overrides={"chrom": pl.Utf8, "start": pl.Int64, "end": pl.Int64},
)

def make_hardmask_checker(chrom, start, end):
    intervals = _bed.filter(
        (pl.col("chrom") == f"chr{chrom}")
        & (pl.col("end")   > start - 1)
        & (pl.col("start") < end)
    ).collect()
    if intervals.height == 0:
        return lambda p: False
    starts = intervals["start"].to_numpy()
    ends   = intervals["end"].to_numpy()
    def check(pos):
        p0 = pos - 1
        return bool(np.any((starts <= p0) & (ends > p0)))
    return check

def load_gtf_exons():
    if os.path.exists(GTF_CACHE):
        return pl.read_parquet(GTF_CACHE)
    if not os.path.exists(GTF_PATH):
        print(f"Downloading {GTF_URL} → {GTF_PATH} …")
        gz_path = GTF_PATH + ".gz"
        urllib.request.urlretrieve(GTF_URL, gz_path)
        with gzip.open(gz_path, "rb") as f_in, open(GTF_PATH, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)
    print(f"Parsing {GTF_PATH} (one-time, ~1 min) …")
    df = pl.read_csv(
        GTF_PATH, separator="\t", has_header=False, comment_prefix="#",
        new_columns=["chrom","source","feature","start","end","score",
                     "strand","frame","attribute"],
        schema_overrides={"chrom": pl.Utf8, "start": pl.Int64, "end": pl.Int64,
                          "strand": pl.Utf8, "attribute": pl.Utf8,
                          "score": pl.Utf8, "frame": pl.Utf8},
    ).filter(pl.col("feature") == "exon")
    df = df.with_columns([
        pl.col("attribute").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_name"),
        pl.col("attribute").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    ]).drop("attribute", "source", "frame", "score", "feature")
    df.write_parquet(GTF_CACHE)
    return df

GTF_EXONS = load_gtf_exons()
n_genes = GTF_EXONS.select(pl.col("gene_name").n_unique()).item()
print(f"GTF: {GTF_EXONS.height:,} exons across {n_genes:,} genes")

def genes_in_region(chrom, start, end):
    sub = GTF_EXONS.filter(
        (pl.col("chrom") == f"chr{chrom}")
        & (pl.col("end") >= start)
        & (pl.col("start") <= end)
    )
    if sub.height == 0:
        return pd.DataFrame(columns=["gene_name","transcript_id","strand",
                                      "tx_start","tx_end","span","exons"])
    tx_span = sub.group_by(["gene_name","transcript_id","strand"]).agg([
        pl.col("start").min().alias("tx_start"),
        pl.col("end").max().alias("tx_end"),
    ]).with_columns((pl.col("tx_end") - pl.col("tx_start")).alias("span"))
    longest = (tx_span.sort("span", descending=True)
                       .group_by("gene_name").first()).to_pandas()
    exons_by_tx = {}
    for tid in longest["transcript_id"]:
        ex = sub.filter(pl.col("transcript_id") == tid)
        exons_by_tx[tid] = list(zip(ex["start"].to_list(), ex["end"].to_list()))
    longest["exons"] = longest["transcript_id"].map(exons_by_tx)
    return longest.sort_values("tx_start").reset_index(drop=True)

def draw_gene_track(ax, chrom, region_start, region_end, max_genes=20):
    genes = genes_in_region(chrom, region_start, region_end)
    if len(genes) == 0:
        ax.text(0.5, 0.5, "(no annotated genes)", transform=ax.transAxes,
                ha="center", va="center", color="gray", fontsize=14)
        ax.set_yticks([])
        ax.set_xlim(region_start / 1e6, region_end / 1e6)
        return
    if len(genes) > max_genes:
        genes = (genes.sort_values("span", ascending=False).head(max_genes)
                       .sort_values("tx_start").reset_index(drop=True))
    min_gap = (region_end - region_start) * 0.24
    rows, row_of = [], []
    for _, g in genes.iterrows():
        placed = False
        for ri, last in enumerate(rows):
            if g["tx_start"] > last + min_gap:
                rows[ri] = g["tx_end"]; row_of.append(ri); placed = True; break
        if not placed:
            rows.append(g["tx_end"]); row_of.append(len(rows) - 1)
    row_height = 0.6
    row_gap = 3.0
    ax.set_ylim(-0.7, (len(rows) - 1) * row_gap + 2.8)
    min_exon_w = (region_end - region_start) * 0.002
    for (_, g), row in zip(genes.iterrows(), row_of):
        y = row * row_gap + 0.5
        ax.hlines(y, g["tx_start"]/1e6, g["tx_end"]/1e6,
                  colors="black", linewidth=2.0, zorder=2)
        for ex_s, ex_e in g["exons"]:
            es = max(ex_s, region_start); ee = min(ex_e, region_end)
            if ee <= es:
                continue
            ax.add_patch(plt.Rectangle(
                (es/1e6, y - row_height / 2),
                max((ee - es)/1e6, min_exon_w/1e6), row_height,
                facecolor="black", edgecolor="none", alpha=0.88, zorder=3))
        center_x = np.clip((g["tx_start"]+g["tx_end"])/2,
                           region_start + (region_end-region_start)*0.05,
                           region_end - (region_end-region_start)*0.05) / 1e6
        ax.text(center_x, y + row_height / 2 + 0.35, g['gene_name'],
                ha="center", va="bottom", fontsize=FONT - 4,
                fontstyle="italic", color="black", zorder=4)
    ax.set_xlim(region_start / 1e6, region_end / 1e6)
    ax.set_ylabel("Genes")
    ax.set_yticks([])
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

def _id_col(columns):
    for c in ("sample_id","person_id","research_id","s","IID"):
        if c in columns:
            return c
    return columns[0]

def _harmonize_pca(df_raw, n_pcs):
    df = df_raw.copy()
    df.rename(columns={_id_col(list(df.columns)): "person_id"}, inplace=True)
    df["person_id"] = df["person_id"].astype(int)
    if "scores" in df.columns and df["scores"].dtype == object:
        exp = df["scores"].str.strip().str.strip("[]").str.split(",", expand=True)
        for i in range(n_pcs):
            df[f"pc{i+1}"] = exp[i].astype(float)
        df.drop(columns=["scores"], inplace=True, errors="ignore")
    else:
        up = {c.upper(): c for c in df.columns if c != "person_id"}
        remap = {}
        for i in range(1, n_pcs + 1):
            for pat in [f"PC{i}", f"SCORES_{i-1}", f"SCORES_{i}"]:
                if pat in up:
                    remap[up[pat]] = f"pc{i}"; break
        if len(remap) < n_pcs:
            raise ValueError(f"Found only {len(remap)}/{n_pcs} PC columns")
        df.rename(columns=remap, inplace=True)
    return df[["person_id"] + [f"pc{i}" for i in range(1, n_pcs + 1)]]

if USE_COVARIATES:
    print("Loading covariates (age/sex + 20 AFR PCs, PheTK sample set) …")
    base = (pl.read_csv(DISC_COV, separator="\t")
            .select(["person_id", "age_at_last_ehr_event",
                     "age_at_last_ehr_event_squared", "sex_at_birth"])
            .drop_nulls().to_pandas())
    pca_raw = pd.read_csv(DISC_PCA, sep="\t", compression="gzip")
    pca_new = _harmonize_pca(pca_raw, NEW_N_PCS)
    if not os.path.exists(FLAGGED_IDS_PATH):
        raise FileNotFoundError(FLAGGED_IDS_PATH)
    flagged_df  = pd.read_csv(FLAGGED_IDS_PATH, sep="\t")
    FLAGGED_IDS = set(pd.to_numeric(flagged_df.iloc[:, 0], errors="coerce")
                      .dropna().astype(int).tolist())
    print(f"  in-cohort flagged related: {len(FLAGGED_IDS)}")
    cov_pd = base.merge(pca_new, on="person_id", how="inner")
    cov_pd = cov_pd[~cov_pd["person_id"].isin(FLAGGED_IDS)].dropna()
    if cov_pd["sex_at_birth"].dtype == object:
        cov_pd["sex_at_birth"] = (cov_pd["sex_at_birth"].astype(str).str.lower()
                                  .map({"male": 1, "m": 1, "female": 0, "f": 0})
                                  .fillna(0).astype(float))
    PC_COLS  = [f"pc{i}" for i in range(1, NEW_N_PCS + 1)]
    COV_COLS = ["age_at_last_ehr_event", "age_at_last_ehr_event_squared",
                "sex_at_birth"] + PC_COLS
    cov_pd = cov_pd.set_index("person_id")
    cov_pd.index = cov_pd.index.astype(str)
    print(f"  cohort: {len(cov_pd)} samples")
else:
    cov_pd, COV_COLS = None, []

_v0 = CyVCF(SV_VCF); SAMPLES = list(_v0.samples); _v0.close()
N_SAMPLES  = len(SAMPLES)
SAMPLE_INT = np.array([int(s) for s in SAMPLES], dtype=np.int64)
print(f"VCF samples: {N_SAMPLES}")

if USE_COVARIATES:
    cov_index = np.full(N_SAMPLES, -1, dtype=np.int64)
    pos_in_cov = {pid: i for i, pid in enumerate(cov_pd.index.tolist())}
    for i, s in enumerate(SAMPLES):
        j = pos_in_cov.get(s)
        if j is not None:
            cov_index[i] = j
    HAS_COV  = cov_index >= 0
    cov_rows = np.where(HAS_COV)[0]
    COV_MAT  = np.full((N_SAMPLES, len(COV_COLS)), np.nan, dtype=float)
    COV_MAT[cov_rows] = cov_pd.iloc[cov_index[cov_rows]][COV_COLS].to_numpy(
        dtype=float)
    print(f"  HAS_COV (PheTK cohort): {HAS_COV.sum()}")
else:
    HAS_COV = np.ones(N_SAMPLES, dtype=bool)
    COV_MAT = None

pc_counts = pl.read_csv(DISC_PHE, separator="\t",
                        schema_overrides={"phecode": pl.Utf8})

def build_phenotype(phecode):
    cases = set(pc_counts.filter(
        (pl.col("phecode") == phecode)
        & (pl.col("count")   >= MIN_PHECODE_COUNT)
    )["person_id"].to_list())
    any_hit = set(pc_counts.filter(
        pl.col("phecode") == phecode
    )["person_id"].to_list())
    y = np.full(N_SAMPLES, np.nan, dtype=float)
    in_case = np.fromiter((pid in cases   for pid in SAMPLE_INT),
                          dtype=bool, count=N_SAMPLES)
    in_any  = np.fromiter((pid in any_hit for pid in SAMPLE_INT),
                          dtype=bool, count=N_SAMPLES)
    y[in_case] = 1.0
    y[(~in_case) & (~in_any)] = 0.0
    if USE_COVARIATES:
        y[~HAS_COV] = np.nan
    return y

def fetch_region(chrom, start, end):
    vcf = CyVCF(SV_VCF)
    hm  = make_hardmask_checker(chrom, start, end)
    ids, pos_list, dosages = [], [], []
    is_sv_list, sv_type_list = [], []
    for rec in vcf(f"chr{chrom}:{max(1, start)}-{end}"):
        if rec.ALT is None or len(rec.ALT) != 1:
            continue
        gt  = rec.gt_types
        dos = np.where(gt == 2, np.nan,
                       np.where(gt == 3, 2.0, gt.astype(float)))
        nz  = ~np.isnan(dos)
        an  = 2 * int(nz.sum())
        if an == 0:
            continue
        if (1.0 - nz.mean()) > MAX_MISS:
            continue
        ac  = float(np.nansum(dos))
        mac = min(ac, an - ac)
        if mac < MIN_MAC:
            continue
        if np.isnan(dos).any():
            dos = np.where(nz, dos, np.nanmean(dos))

        ref, alt = rec.REF, rec.ALT[0]
        diff = len(ref) - len(alt)
        if diff >= SV_LEN_DIFF:
            is_sv, sv_type = True, "DEL"
        elif -diff >= SV_LEN_DIFF:
            is_sv, sv_type = True, "INS"
        else:
            is_sv, sv_type = False, ""

        if (not is_sv) and len(ref) == 1 and len(alt) == 1 and hm(rec.POS):
            continue

        if is_sv:
            svlen = rec.INFO.get("SVLEN", "")
            vid = f"chr{chrom}:{rec.POS}:{sv_type}:{svlen}"
        else:
            vid = f"chr{chrom}:{rec.POS}:{ref}:{alt}"

        ids.append(vid)
        pos_list.append(rec.POS)
        dosages.append(dos.astype(np.float32))
        is_sv_list.append(bool(is_sv))
        sv_type_list.append(sv_type)
    vcf.close()
    if not dosages:
        return [], np.zeros((N_SAMPLES, 0), dtype=np.float32), [], [], []
    return ids, np.stack(dosages, axis=1), pos_list, is_sv_list, sv_type_list

def residualize(y, Z):
    Z1 = np.column_stack([np.ones(len(y)), Z])
    coef, *_ = np.linalg.lstsq(Z1, y, rcond=None)
    return y - Z1 @ coef

def score_test_region(Xm, ym, Zm):
    Z1       = np.column_stack([np.ones(len(ym)), Zm])
    null_fit = sm.Logit(ym, Z1).fit(disp=False, maxiter=300, method="newton")
    p_null   = null_fit.predict(Z1)
    W        = p_null * (1 - p_null)
    r_null   = ym - p_null
    WZ       = Z1 * W[:, None]
    ZtWZ_inv = np.linalg.pinv(Z1.T @ WZ)
    xtwx     = (Xm * (Xm * W[:, None])).sum(axis=0)
    xtwz     = Xm.T @ WZ
    adj      = np.einsum("pk,kl,pl->p", xtwz, ZtWZ_inv, xtwz)
    info     = xtwx - adj
    U        = Xm.T @ r_null
    valid    = info > 0
    chi2     = np.where(valid, (U ** 2) / np.where(valid, info, np.nan), np.nan)
    return (np.where(valid, U / np.where(valid, info, np.nan), np.nan),
            np.where(valid, 1.0 / np.sqrt(np.where(valid, info, np.nan)), np.nan),
            stats.chi2.sf(chi2, df=1))

def wald_logit_single(x, y, Z):
    design = sm.add_constant(np.column_stack([x, Z]))
    try:
        fit = sm.Logit(y, design).fit(disp=False, maxiter=300, method="newton")
    except Exception:
        try:
            fit = sm.Logit(y, design).fit(disp=False, maxiter=500, method="bfgs")
        except Exception:
            return None
    return dict(beta=float(fit.params[1]), se=float(fit.bse[1]),
                p=float(fit.pvalues[1]), or_=float(np.exp(fit.params[1])),
                n=int(fit.nobs))

def variant_allele_stats(dos):
    n = len(dos)
    n_carriers = int((dos > 0).sum())
    n_hom      = int((dos == 2).sum())
    mac = 2 * n_hom + (n_carriers - n_hom)
    af  = mac / (2 * n) if n else float("nan")
    return n_carriers, mac, af

cov_suffix     = "cov20" if USE_COVARIATES else "nocov"
summary_rows   = []
cs_member_rows = []

for ti, t in tasks.iterrows():
    tag     = f"{t.CHR}_{t.POS}_{t.REF}_{t.ALT}__{t.phecode}__{cov_suffix}"
    out_tsv = os.path.join(WORKDIR, f"susie_{tag}.tsv")
    out_fig = os.path.join(WORKDIR, f"locus_{tag}.png")

    gene_str = (str(t.get("gene_in_window", ""))
                if pd.notna(t.get("gene_in_window")) else "")
    print(f"\n[{ti+1}/{len(tasks)}] {t.variant_tag}  {t.phecode} "
          f"{t.phecode_name}  ({gene_str})")

    y_all   = build_phenotype(t.phecode)
    mask    = ~np.isnan(y_all)
    n_cases = int(np.nansum(y_all))
    n_ctrl  = int(mask.sum() - n_cases)
    if n_cases < MIN_CASES or n_ctrl < MIN_CASES:
        print(f"  skip: cases={n_cases} controls={n_ctrl}")
        continue

    start = max(1, int(t.POS) - WINDOW)
    end   = int(t.POS) + WINDOW
    var_ids, X, positions, is_sv_list, sv_type_list = fetch_region(
        int(t.CHR), start, end)
    if X.shape[1] == 0:
        print("  skip: no variants in region")
        continue

    Xm = X[mask].astype(float)
    ym = y_all[mask].astype(float)
    Zm = COV_MAT[mask].astype(float) if USE_COVARIATES else None

    sv_arr      = np.array(is_sv_list, dtype=bool)
    sv_type_arr = np.array(sv_type_list)
    n_ins = int((sv_type_arr == "INS").sum())
    n_del = int((sv_type_arr == "DEL").sum())
    print(f"  variants={Xm.shape[1]}  SVs={int(sv_arr.sum())} "
          f"(INS={n_ins}, DEL={n_del})  n={len(ym)} "
          f"cases={n_cases} ctrls={n_ctrl}")

    lead_idx = None
    lead_vid = f"chr{t.CHR}:{t.POS}:{t.REF}:{t.ALT}"
    for i, vid in enumerate(var_ids):
        if vid == lead_vid:
            lead_idx = i; break

    if lead_idx is None:
        for i, (p, sv) in enumerate(zip(positions, is_sv_list)):
            if p == int(t.POS) and not sv:
                lead_idx = i; break
    if lead_idx is None:
        print(f"  ⚠ lead SNV {lead_vid} not found in VCF region — skipping")
        continue

    y_use = residualize(ym, Zm) if USE_COVARIATES else (ym - ym.mean())
    try:
        pip, cs_out = run_susie(Xm, y_use, L=L_SUSIE)
    except Exception as exc:
        print(f"  susie failed: {exc}")
        continue

    Z_use = Zm if USE_COVARIATES else np.zeros((len(ym), 1))
    beta_s, se_s, p_s = score_test_region(Xm, ym, Z_use)
    nlp = -np.log10(np.clip(p_s, 1e-300, 1.0))

    Xc  = Xm - Xm.mean(axis=0)
    lc  = Xc[:, lead_idx]
    num = Xc.T @ lc
    den = np.sqrt((Xc ** 2).sum(axis=0) * (lc ** 2).sum())
    r2  = (num / np.where(den > 0, den, np.nan)) ** 2

    wald_lead = wald_logit_single(Xm[:, lead_idx], ym, Z_use)
    lead_car, lead_mac, lead_af = variant_allele_stats(Xm[:, lead_idx])

    df = pd.DataFrame({
        "variant_id":    var_ids,
        "chrom":         [f"chr{t.CHR}"] * len(var_ids),
        "pos":           positions,
        "is_sv":         is_sv_list,
        "sv_type":       sv_type_list,
        "pip":           pip,
        "cs":            [""] * len(var_ids),
        "beta_score":    beta_s,
        "se_score":      se_s,
        "p_score":       p_s,
        "neg_log_p":     nlp,
        "r2_to_lead":    r2,
    })
    for cs_name, idxs in cs_out.items():
        for j in idxs:
            prev = df.at[j, "cs"]
            df.at[j, "cs"] = f"{prev},{cs_name}" if prev else cs_name
    df["lead_variant"] = False
    df.at[lead_idx, "lead_variant"] = True
    df.to_csv(out_tsv, sep="\t", index=False)

    cs_member_idxs = sorted({j for idxs in cs_out.values() for j in idxs})
    for j in cs_member_idxs:
        w = wald_logit_single(Xm[:, j], ym, Z_use)
        cs_member_rows.append({
            "variant_tag":   t.variant_tag,
            "lead_chrom":    int(t.CHR),
            "lead_pos":      int(t.POS),
            "phecode":       t.phecode,
            "phecode_name":  t.phecode_name,
            "cs":            df.at[j, "cs"],
            "variant_id":    var_ids[j],
            "pos":           positions[j],
            "is_sv":         bool(is_sv_list[j]),
            "sv_type":       sv_type_list[j],
            "is_lead":       (j == lead_idx),
            "pip":           float(pip[j]),
            "r2_to_lead":    float(r2[j]) if np.isfinite(r2[j]) else np.nan,
            "beta_score":    float(beta_s[j]),
            "se_score":      float(se_s[j]),
            "p_score":       float(p_s[j]),
            "beta_wald":     w["beta"] if w else np.nan,
            "se_wald":       w["se"]   if w else np.nan,
            "p_wald":        w["p"]    if w else np.nan,
            "or_wald":       w["or_"]  if w else np.nan,
        })

    lead_pip   = float(pip[lead_idx])
    lead_in_cs = bool(df.at[lead_idx, "cs"] != "")
    n_cs       = len(cs_out)
    cs_sizes   = ";".join(str(len(v)) for v in cs_out.values())
    cs_total   = sum(len(v) for v in cs_out.values())
    cs_has_sv  = sum(1 for idxs in cs_out.values()
                     if any(is_sv_list[j] for j in idxs))

    top_pip_idx = int(np.argmax(pip))
    top_pip_var = var_ids[top_pip_idx]
    top_pip_val = float(pip[top_pip_idx])

    print(f"  ─ lead SNV ─")
    if wald_lead:
        print(f"    {lead_vid}  carriers={lead_car}  AF={lead_af:.4f}  "
              f"beta={wald_lead['beta']:+.4f}  p={wald_lead['p']:.2e}  "
              f"PIP={lead_pip:.3f}  in_CS={lead_in_cs}")
    print(f"    top PIP variant: {top_pip_var}  PIP={top_pip_val:.3f}  "
          f"is_sv={is_sv_list[top_pip_idx]}")
    if n_cs:
        print(f"  ─ {n_cs} credible set(s)  sizes={cs_sizes}  "
              f"({cs_has_sv} contain ≥1 SV) ─")
        for name in sorted(cs_out):
            for j in sorted(cs_out[name], key=lambda j: -pip[j]):
                kind = sv_type_list[j] if is_sv_list[j] else "SNV"
                ld   = f"r²={r2[j]:.3f}" if np.isfinite(r2[j]) else "r²=NA"
                flag = "  ← LEAD" if j == lead_idx else ""
                print(f"    CS {name}  [{kind:3s}] "
                      f"chr{t.CHR}:{positions[j]:>10}  "
                      f"PIP={pip[j]:.3f}  p={p_s[j]:.2e}  {ld}  "
                      f"{var_ids[j]}{flag}")
    else:
        print("  ─ no credible sets ─")

    pos_mb = np.array(positions) / 1e6
    cs_palette = plt.cm.tab10.colors

    ins_mask = sv_arr & (sv_type_arr == "INS")
    del_mask = sv_arr & (sv_type_arr == "DEL")
    snv_mask = ~sv_arr.copy()
    snv_mask[lead_idx] = False

    fig = plt.figure(figsize=(22, 11))
    gs  = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.03],
                           height_ratios=[5, 1.6], hspace=0.06, wspace=0.18)
    ax_pip = fig.add_subplot(gs[0, 0])
    ax_pv  = fig.add_subplot(gs[0, 1])
    cax    = fig.add_subplot(gs[:, 2])
    ax_ga  = fig.add_subplot(gs[1, 0], sharex=ax_pip)
    ax_gb  = fig.add_subplot(gs[1, 1], sharex=ax_pv)

    in_any_cs = df["cs"] != ""
    ax_pip.scatter(pos_mb[~in_any_cs.values & snv_mask],
                   pip[~in_any_cs.values & snv_mask],
                   s=45, alpha=0.35, color="gray", label="SNV (not in CS)")
    for k, cs_name in enumerate(cs_out):
        m = df["cs"].str.contains(cs_name, regex=False).values & in_any_cs.values
        if m.any():
            ax_pip.scatter(pos_mb[m], pip[m], s=130,
                           color=cs_palette[k % 10],
                           edgecolor="black", linewidth=0.6,
                           label=f"CS {cs_name} (n={len(cs_out[cs_name])})")
    if ins_mask.any():
        ax_pip.scatter(pos_mb[ins_mask], pip[ins_mask], s=200, marker="^",
                       facecolors="none", edgecolor="red", linewidth=1.5,
                       label="INS (SV)")
    if del_mask.any():
        ax_pip.scatter(pos_mb[del_mask], pip[del_mask], s=200, marker="v",
                       facecolors="none", edgecolor="red", linewidth=1.5,
                       label="DEL (SV)")
    ax_pip.scatter([pos_mb[lead_idx]], [pip[lead_idx]], s=400, marker="D",
                   facecolors="gold", edgecolor="black", linewidth=2.0,
                   zorder=6, label="lead SNV")
    ax_pip.axvline(int(t.POS) / 1e6, color="gold", ls=":", alpha=0.6, lw=1.5)
    ax_pip.set_ylabel("PIP")
    ax_pip.set_ylim(-0.02, 1.05)
    ax_pip.set_title("SuSiE PIP")
    ax_pip.legend(loc="upper right", framealpha=0.9, fontsize=14)
    ax_pip.tick_params(labelbottom=False)

    finite_r2 = np.where(np.isfinite(r2), r2, 0.0)
    order = np.argsort(finite_r2[snv_mask])
    sc = ax_pv.scatter(pos_mb[snv_mask][order], nlp[snv_mask][order],
                       s=70, c=finite_r2[snv_mask][order], cmap="viridis",
                       vmin=0, vmax=1, alpha=0.9, edgecolor="none")
    if ins_mask.any():
        ax_pv.scatter(pos_mb[ins_mask], nlp[ins_mask], s=200, marker="^",
                      c=finite_r2[ins_mask], cmap="viridis", vmin=0, vmax=1,
                      alpha=0.9, edgecolor="none", label="INS")
    if del_mask.any():
        ax_pv.scatter(pos_mb[del_mask], nlp[del_mask], s=200, marker="v",
                      c=finite_r2[del_mask], cmap="viridis", vmin=0, vmax=1,
                      alpha=0.9, edgecolor="none", label="DEL")
    ax_pv.scatter([pos_mb[lead_idx]], [nlp[lead_idx]], s=400, marker="D",
                  facecolors="gold", edgecolor="black", linewidth=2.0,
                  zorder=6, label="lead SNV")
    ax_pv.axvline(int(t.POS) / 1e6, color="gold", ls=":", alpha=0.6, lw=1.5)
    ax_pv.set_ylabel(r"$-\log_{10}$(p)")
    ax_pv.set_title("Score-test p  (color = r² to lead)")
    ax_pv.legend(loc="upper right", fontsize=14)
    ax_pv.tick_params(labelbottom=False)
    cbar = fig.colorbar(sc, cax=cax)
    cbar.set_label("r² to lead SNV")
    cbar.ax.tick_params(labelsize=FONT - 4)

    for ax_g_ in (ax_ga, ax_gb):
        draw_gene_track(ax_g_, int(t.CHR), start, end)
        ax_g_.axvline(int(t.POS) / 1e6, color="gold", ls=":", alpha=0.6, lw=1.5)
        ax_g_.set_xlabel(f"chr{t.CHR} position (Mb)")

    lead_p_str  = f"  lead p={wald_lead['p']:.2g}" if wald_lead else ""
    meta_p_str  = f"  p_meta={t.p_meta:.2g}" if pd.notna(t.get("p_meta")) else ""
    fig.suptitle(
        f"chr{t.CHR}:{t.POS} {t.REF}>{t.ALT}"
        f"{(' — '+gene_str) if gene_str else ''}"
        f" → {t.phecode} {t.phecode_name}\n"
        f"n={int(mask.sum())} cases={n_cases}{lead_p_str}{meta_p_str}",
        fontsize=FONT)
    fig.savefig(out_fig, dpi=120, bbox_inches="tight")
    plt.show()

    summary_rows.append({
        "variant_tag":       t.variant_tag,
        "chrom":             int(t.CHR),
        "pos":               int(t.POS),
        "ref":               t.REF,
        "alt":               t.ALT,
        "phecode":           t.phecode,
        "phecode_name":      t.phecode_name,
        "gene_in_window":    gene_str,
        "n_samples":         int(mask.sum()),
        "n_cases":           n_cases,
        "n_controls":        n_ctrl,
        "n_variants":        int(Xm.shape[1]),
        "n_svs":             int(sv_arr.sum()),
        "n_ins":             n_ins,
        "n_del":             n_del,
        "lead_carriers":     lead_car,
        "lead_mac":          lead_mac,
        "lead_af":           lead_af,
        "lead_beta_wald":    wald_lead["beta"] if wald_lead else np.nan,
        "lead_se_wald":      wald_lead["se"]   if wald_lead else np.nan,
        "lead_p_wald":       wald_lead["p"]    if wald_lead else np.nan,
        "lead_or_wald":      wald_lead["or_"]  if wald_lead else np.nan,
        "lead_pip":          lead_pip,
        "lead_in_cs":        lead_in_cs,
        "top_pip_variant":   top_pip_var,
        "top_pip":           top_pip_val,
        "top_pip_is_sv":     bool(is_sv_list[top_pip_idx]),
        "top_pip_sv_type":   sv_type_list[top_pip_idx],
        "top_pip_pos":       int(positions[top_pip_idx]),
        "top_pip_p_score":   float(p_s[top_pip_idx]),
        "n_credible_sets":   n_cs,
        "cs_sizes":          cs_sizes,
        "cs_members_total":  cs_total,
        "n_cs_with_sv":      cs_has_sv,
        "p_meta":            t.get("p_meta", np.nan),
        "beta_meta":         t.get("beta_meta", np.nan),
        "p_disc":            t.get("p_d", np.nan),
        "beta_disc":         t.get("beta_d", np.nan),
    })

if summary_rows:
    summary = pd.DataFrame(summary_rows)
    sum_path = os.path.join(WORKDIR, f"susie_meta_snv_summary_{cov_suffix}.tsv")
    summary.to_csv(sum_path, sep="\t", index=False)
    print(f"\n✓ Per-task summary → {sum_path}  ({len(summary)} tasks)")

    cs_detail = pd.DataFrame(cs_member_rows)
    cs_path = os.path.join(WORKDIR, f"susie_meta_snv_cs_members_{cov_suffix}.tsv")
    cs_detail.to_csv(cs_path, sep="\t", index=False)
    print(f"✓ CS-member detail → {cs_path}  ({len(cs_detail)} rows)")

    show_cols = [
        "chrom", "pos", "ref", "alt", "phecode", "phecode_name",
        "gene_in_window", "n_cases", "lead_carriers", "lead_af",
        "lead_p_wald", "lead_pip", "lead_in_cs",
        "top_pip_variant", "top_pip",
        "n_credible_sets", "cs_sizes", "n_cs_with_sv",
        "p_meta", "p_disc",
    ]
    show_cols = [c for c in show_cols if c in summary.columns]
    print(f"\nKey results (lead SNV fine-mapping vs meta/disc p-values):")
    with pd.option_context("display.max_colwidth", 32, "display.width", 280,
                            "display.float_format",
                            lambda x: f"{x:.3g}" if pd.notna(x) else "—"):
        print(summary[show_cols].to_string(index=False))
else:
    print("\nNo tasks completed.")
